# Mathrix Souls

RPG de fantasia sombria em que **as matrizes são a sua magia**, pensado para ser jogado **sem olhar a tela**: tudo é narrado, e o jogador responde pelo teclado.

O notebook tem o **jogo completo**. No menu, o jogador escolhe:
- **DEMO** (cenas 1 a 9): encontro com o Alquimista, combate com o Goblin e travessia de barco, encerrando no Farol.
- **Jogo completo** (cenas 1 a 12 + epílogo): continua do Farol para a masmorra, os guardiões e o Dragão.

| Cena | O que acontece | Matemática |
|---|---|---|
| 1. A Casa | O jovem mago acorda | — |
| 2. A Taverna | Max bêbado e o Alquimista propõe o acordo | — |
| 3. A Lição | Alquimista ensina a matriz quadrada e a soma | Soma de matrizes |
| 4. O Grilo Falante | O guia aparece e treina os comandos | — |
| 5. A Bifurcação | Pássaros (floresta) ou silêncio (caminho vermelho) | — |
| 6. O Goblin | Esquiva pelo som + pedágio da soma | Soma de matrizes |
| 7. A Praia | Checkpoint e lição da triangular | Triangular superior |
| 8. O Barco | Tempestade: 3 relâmpagos | Identificar a triangular superior |
| 9. O Farol | Chegada à ilha. **Fim da DEMO**. No jogo completo, vira checkpoint e o Alquimista ensina mais duas magias | Diagonal, identidade e "o tipo mais específico" |
| 10. A Masmorra | A história do farol e a escolha do corredor | — |
| 11A. Cavaleiro Fantasma | Banquete, teste do zero e o espelho | Nula (A + 0 = A), transposta e simétrica |
| 11B. Golem de Pedra | Pergunta do Golem e o puzzle das pedras | Nula (A − A = 0); pedras nula, identidade e diagonal |
| 12. O Dragão | Garras + classificar o fogo + golpe final | Classificar pelo tipo mais específico; M − M = 0 |
| Epílogo | De volta à taverna | — |

## Como rodar
- **Para jogar:** dê dois cliques em `MathrixSouls.exe`, na pasta principal do projeto. Ele roda este notebook sem precisar de Jupyter nem de instalação.
- **No Jupyter/VS Code:** rode as células de cima para baixo (*Run All*). A célula **▶ Jogar** inicia o jogo. Digite a tecla e aperte **Enter** em cada resposta. (No `.exe` não precisa de Enter, e dá para responder no meio da fala.)
- **Voz padrão:** enquanto uma fala não estiver gravada, ela sai na **voz da Maria** (voz do Windows em português). O jogo já é jogável sem nenhuma gravação.
- **Vozes gravadas:** cada fala tem um código (ex.: `A02`). Grave e salve como `audios/A02.wav`, na mesma pasta deste notebook (`arquivos/audios`). Efeitos sonoros vão em `audios/sons/<nome>.wav`.
  - Formatos: `.wav`, `.mp3` ou `.ogg` (para `.mp3`/`.ogg` no notebook, rode `pip install pygame-ce`; os `.exe` já incluem). **Recomendado: `.mp3` ou `.ogg`, mono**, que ocupam bem menos espaço.
  - O **Apêndice**, no fim do notebook, lista todas as falas a gravar e mostra quais ainda faltam.

### Gravação
- **Uma fala por arquivo**, com o nome exato do código (`N10.wav`, `GB03.wav`, `S02_1.wav`...).
- Falas com `_` no código são **variações** (ex.: `S02_2` = "restam duas vidas", `S02_1` = "resta uma vida"). Grave todas.
- As matrizes são sorteadas na hora, então elas são lidas **juntando pedaços gravados** (`NUM_0` a `NUM_9`, `MAT_EM_CIMA`, `OPCAO_A`...). Esses pedaços também estão no Apêndice. Grave curto e sem pausa no começo ou no fim, para a frase montada soar natural.

## Comandos (ASDW)
| Tecla | Ação |
|---|---|
| **W** | Frente / confirmar / opção do meio |
| **A** | Esquerda / primeira opção |
| **S** | Trás / recuar |
| **D** | Direita / terceira opção |
| **R** | Repetir a última fala (ou a pergunta inteira) |
| **P** | Pausar |
| **V** | Ouvir quantas vidas restam |
| **O** | Ouvir só as opções de agora |
| **H** | Ouvir todos os comandos |
| **Q** | Sair do jogo |

## Estrutura do notebook
- **Motor** (configuração, voz, comandos, estado, matemática): o "sistema" do jogo.
- **Falas do Narrador e do sistema**: nome do jogo, menu DEMO/completo, instruções, vidas, pausa.
- **Uma seção por cena**: um bloco com as **falas** (cada fala tem um código, que também é o nome do arquivo de áudio) e um bloco com a **lógica** da cena.
- **Menu e início do jogo** no final.

---
# Parte 1 — Motor do jogo
## 1.1 Configuração

In [ ]:
import os
import random
import time

import numpy as np

VELOCIDADE = 1.0       # multiplica as pausas entre falas (0 = sem pausa)
VOZ_ATIVA = True       # voz sintetizada para as falas que ainda não foram gravadas
usar_gravacoes = True  # menu, tecla G: True = vozes gravadas (a do computador completa as que faltam);
                       #                False = só as vozes do computador. Efeitos sonoros tocam sempre.
VOZ_PREFERIDA = "Maria"  # voz feminina (Narrador, Grilo...): Microsoft Maria
VOZ_MASCULINA = "Daniel"  # voz masculina (Alquimista, Goblin, Dragão...): Microsoft Daniel, se existir
VOZ_VELOCIDADE = 1     # velocidade base da voz: de -10 (lenta) a 10 (rápida)
VELOCIDADES_DA_FALA = [0.75, 1.0, 1.25, 1.5, 2.0, 2.5, 3.0]   # A diminui, D aumenta. 3x é o máximo da voz do Windows
velocidade_da_fala = 1.0   # a escolhida no menu; acelera a voz sintetizada e as pausas
DIZER_NOME_DO_PERSONAGEM = True  # voz sintetizada diz quem fala ("Alquimista.") quando o personagem muda
SINAL_DE_ESPERA = True  # som curto quando o jogo espera uma tecla
BIPE_DE_VELOCIDADE = True  # + e - fora de uma fala: bipe curto (agudo = acelerou, grave = desacelerou)
PODE_INTERROMPER = True  # apertar uma tecla durante a fala para a voz (e, numa pergunta, já vale como resposta)
PASTA_AUDIOS = "audios"  # falas gravadas: audios/<CODIGO>.wav  |  efeitos: audios/sons/<nome>.wav
SEMENTE = None         # número fixo = partidas sempre iguais (bom para testar)
MODO_TESTE = False     # True = as teclas vêm da lista ENTRADAS_TESTE, sem digitar
ENTRADAS_TESTE = []

random.seed(SEMENTE)

## 1.2 Voz e áudio
Toda fala passa por `anunciar()`: imprime o texto e toca a **gravação** (`audios/<CÓDIGO>.wav`).
Enquanto a gravação não existir, a fala sai na **voz da Maria** (voz do Windows em português), configurada em `VOZ_PREFERIDA`. Assim o jogo já é jogável sem nenhuma gravação.

A voz é ligada na primeira forma que funcionar:
1. **SAPI direto** (pacote `comtypes`, que vem com o `pyttsx3`): a mais rápida.
2. **PowerShell**: já vem em todo Windows, então não precisa instalar nada.
3. **pyttsx3**: para quem rodar fora do Windows.
4. Nenhuma: o jogo mostra só o texto.

Para tocar as gravações, o jogo usa o **`pygame`** (pacote `pygame-ce`, já incluído nos `.exe`), que aceita `.wav`, `.mp3` e `.ogg`. Se ele não estiver instalado, usa o `winsound`, que já vem no Windows e só toca `.wav`.

**Velocidade da fala (menu, tecla W):** abre o ajuste; **D** aumenta, **A** diminui e **W** volta ao menu. Opções: 0,75x (mais lenta), 1x, 1,25x, 1,5x, 2x, 2,5x e 3x. **3x é o limite**: acima disso, a voz do Windows começa a engolir palavras. Acelera a voz sintetizada (sem mudar o tom) e as pausas entre as falas.
As **gravações** continuam em 1x: acelerar um áudio gravado sem deixar a voz fina pede um processamento à parte (*time-stretch*), que ainda não está no jogo.

In [ ]:
import base64
import subprocess
import sys
from xml.sax.saxutils import escape

try:
    import winsound  # já vem com o Python no Windows (só .wav)
except ImportError:
    winsound = None

try:
    import msvcrt    # já vem com o Python no Windows: lê uma tecla sem precisar de Enter
except ImportError:
    msvcrt = None

os.environ.setdefault("PYGAME_HIDE_SUPPORT_PROMPT", "1")   # esconde a mensagem de boas-vindas do pygame
try:
    import pygame    # toca .wav, .mp3 e .ogg (pip install pygame-ce)
    pygame.mixer.init()
except Exception:
    pygame = None    # sem pygame: só .wav, pelo winsound

EXTENSOES = [".wav", ".mp3", ".ogg"] if pygame else [".wav"]

# Se esta célula rodar de novo no Jupyter, encerra a voz anterior
if globals().get("_modo_voz") == "powershell":
    try:
        _voz.stdin.close()
    except Exception:
        pass

tecla_antecipada = None   # tecla apertada durante uma fala (ainda não usada)

SETAS = {"H": "W", "K": "A", "P": "S", "M": "D"}   # seta para cima/esquerda/baixo/direita


def console_interativo():
    """True no .exe e no terminal (dá para ler tecla na hora); False no Jupyter e nos testes."""
    return (msvcrt is not None and not MODO_TESTE
            and sys.stdin is not None and hasattr(sys.stdin, "isatty") and sys.stdin.isatty())


def ler_tecla_console():
    """Lê uma tecla sem Enter. As setas viram W, A, S, D; Enter vira ""."""
    ch = msvcrt.getwch()
    if ch in ("\x00", "\xe0"):            # teclas especiais (setas) vêm em dois pedaços
        return SETAS.get(msvcrt.getwch(), "?")
    if ch == "\x03":                       # Ctrl+C
        raise KeyboardInterrupt
    if ch in ("\r", "\n"):
        return ""
    return ch.upper()


def acompanhar(terminou, parar, ao_mudar_velocidade=None):
    """Espera o áudio acabar, vigiando o teclado.
    - + ou -: muda a velocidade NA HORA, sem parar nem recomeçar a fala.
    - Outra tecla: para o áudio e guarda a tecla (numa pergunta, ela vira a resposta).
    `terminou()` deve esperar um pouquinho e dizer se o áudio acabou."""
    global tecla_antecipada
    vigiar = console_interativo()
    while not terminou():
        if vigiar and tecla_antecipada is None and msvcrt.kbhit():
            tecla = ler_tecla_console()
            velocidade = globals().get("TECLAS_VELOCIDADE", {})
            if tecla in velocidade:
                if ajustar_velocidade(velocidade[tecla], anunciar_mudanca=False) and ao_mudar_velocidade:
                    ao_mudar_velocidade()
                continue
            if not PODE_INTERROMPER:
                continue
            tecla_antecipada = tecla
            parar()
            return


_voz = None        # o motor de voz em uso
_modo_voz = None   # "sapi", "powershell", "pyttsx3" ou None (sem voz, só texto)


def _indice_voz(nomes):
    """Escolhe a voz: primeiro VOZ_PREFERIDA (Maria), depois qualquer uma em português."""
    for criterio in (VOZ_PREFERIDA.lower(), "portug", "brazil", "pt-br"):
        for i, nome in enumerate(nomes):
            if criterio in nome.lower():
                return i
    return None


TOKENS_DE_VOZ = {"feminina": None, "masculina": None}   # vozes do Windows encontradas


def _todas_as_vozes(voz):
    """Vozes SAPI comuns + as vozes "OneCore" (as do Narrador do Windows, ex.: Microsoft Daniel),
    que não aparecem na lista comum mas funcionam pelo mesmo caminho."""
    import comtypes.client
    tokens = []
    lista = voz.GetVoices()
    tokens += [lista.Item(n) for n in range(lista.Count)]
    try:
        categoria = comtypes.client.CreateObject("SAPI.SpObjectTokenCategory")
        categoria.SetId(r"HKEY_LOCAL_MACHINE\SOFTWARE\Microsoft\Speech_OneCore\Voices", False)
        lista = categoria.EnumerateTokens()
        tokens += [lista.Item(n) for n in range(lista.Count)]
    except Exception:
        pass
    return tokens


def _iniciar_sapi():
    """Voz do Windows direto (rápida). Precisa do pacote comtypes."""
    import comtypes.client
    voz = comtypes.client.CreateObject("SAPI.SpVoice")
    tokens = _todas_as_vozes(voz)
    nomes = [t.GetDescription() for t in tokens]
    i = _indice_voz(nomes)
    if i is not None:
        TOKENS_DE_VOZ["feminina"] = tokens[i]
        voz.Voice = tokens[i]
    masc = next((t for t, n in zip(tokens, nomes) if VOZ_MASCULINA.lower() in n.lower()), None)
    TOKENS_DE_VOZ["masculina"] = masc
    voz.Rate = VOZ_VELOCIDADE
    return voz


# Reserva sem instalar nada: o PowerShell (que vem em todo Windows) fala o texto.
# Cada fala chega em SSML (com tom e ritmo do personagem), em base64 para os acentos chegarem certos.
_SCRIPT_POWERSHELL = r"""
$ErrorActionPreference = 'Stop'
Add-Type -AssemblyName System.Speech
$s = New-Object System.Speech.Synthesis.SpeechSynthesizer
$vozes = @($s.GetInstalledVoices() | ForEach-Object { $_.VoiceInfo })
$v = $vozes | Where-Object { $_.Name -like '*__NOME__*' } | Select-Object -First 1
if (-not $v) { $v = $vozes | Where-Object { $_.Culture.Name -like 'pt*' } | Select-Object -First 1 }
if ($v) { $s.SelectVoice($v.Name) }
$s.Rate = __VELOCIDADE__
[Console]::Out.WriteLine('PRONTO'); [Console]::Out.Flush()
while ($null -ne ($linha = [Console]::In.ReadLine())) {
    $s.SpeakSsml([Text.Encoding]::UTF8.GetString([Convert]::FromBase64String($linha)))
    [Console]::Out.WriteLine('OK'); [Console]::Out.Flush()
}
"""


def _iniciar_powershell():
    script = (_SCRIPT_POWERSHELL.replace("__NOME__", VOZ_PREFERIDA.replace("'", "''"))
              .replace("__VELOCIDADE__", str(int(VOZ_VELOCIDADE))))
    comando = base64.b64encode(script.encode("utf-16-le")).decode("ascii")
    processo = subprocess.Popen(
        ["powershell", "-NoProfile", "-NonInteractive", "-EncodedCommand", comando],
        stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL,
        text=True, creationflags=getattr(subprocess, "CREATE_NO_WINDOW", 0))
    for linha in processo.stdout:
        if linha.strip() == "PRONTO":
            return processo
    processo.kill()
    raise RuntimeError("PowerShell sem voz")


def iniciar_voz():
    """Liga a voz sintetizada. Ordem: SAPI direto > PowerShell > pyttsx3 > só texto."""
    global _voz, _modo_voz
    if not VOZ_ATIVA:
        return
    tentativas = [("sapi", _iniciar_sapi)]
    if sys.platform == "win32":
        tentativas.append(("powershell", _iniciar_powershell))
    tentativas.append(("pyttsx3", lambda: __import__("pyttsx3")))
    for modo, iniciar in tentativas:
        try:
            _voz = iniciar()
            _modo_voz = modo
            return
        except Exception:
            continue
    print("(Voz sintetizada indisponível: o jogo vai mostrar só o texto.)")


def rate_sapi():
    """Velocidade para o SAPI (de -10 a 10): a base VOZ_VELOCIDADE mais a velocidade escolhida
    pelo jogador. Cada ponto acelera ~1,116x (10 pontos = 3x)."""
    import math
    fator = globals().get("velocidade_da_fala", 1.0)
    return max(-10, min(10, VOZ_VELOCIDADE + round(10 * math.log(fator) / math.log(3))))


def voz_do_personagem(quem):
    """(tom, ritmo, "feminina"/"masculina") do personagem, pela tabela VOZES_PERSONAGENS.
    Se o computador não tiver a voz masculina, usa a feminina com o tom de reserva."""
    cfg = globals().get("VOZES_PERSONAGENS", {}).get(quem, (0, 0))
    tom, ritmo = cfg[0], cfg[1]
    genero = cfg[2] if len(cfg) > 2 else "feminina"
    if genero == "masculina" and TOKENS_DE_VOZ.get("masculina") is None:
        tom, genero = cfg[3], "feminina"
    return tom, ritmo, genero


def falar_em_voz(texto, quem=None, nome=None):
    """Fala o texto com o tom e o ritmo do personagem `quem` (tabela VOZES_PERSONAGENS).
    Se `nome` vier, ele é dito antes, na voz do Narrador (ex.: "Alquimista.")."""
    fator = globals().get("velocidade_da_fala", 1.0)
    tom, ritmo, genero = voz_do_personagem(quem)
    tom_n, ritmo_n, genero_n = voz_do_personagem(globals().get("NARRADOR"))   # o nome sai na voz do Narrador
    if _modo_voz == "sapi":
        def falar_trecho(t, tm, rt, gen):
            # Tom e ritmo vão no texto (XML); a velocidade escolhida pelo jogador vai na propriedade
            # Rate, que o Windows aplica na hora, mesmo no meio da fala.
            # Trava: velocidade + ritmo não passa de 10 (acima disso, a voz engole palavras).
            token = TOKENS_DE_VOZ.get(gen) or TOKENS_DE_VOZ.get("feminina")
            if token is not None:
                _voz.Voice = token
            rate_seguro = lambda: min(rate_sapi(), 10 - rt)
            _voz.Rate = rate_seguro()
            xml = f'<pitch absmiddle="{tm}"><rate speed="{rt}">{escape(t)}</rate></pitch>'
            _voz.Speak(xml, 8 | 1)          # 8 = tem marcações de tom/ritmo; 1 = não trava o jogo
            acompanhar(lambda: _voz.WaitUntilDone(30),
                       lambda: _voz.Speak("", 2),                      # 2 = cala na hora
                       lambda: setattr(_voz, "Rate", rate_seguro()))   # + ou -: muda na hora
        if nome:
            falar_trecho(nome + ".", tom_n, ritmo_n, genero_n)
            if tecla_antecipada is not None:     # o jogador já apertou uma tecla durante o nome
                return
        falar_trecho(texto, tom, ritmo, genero)
    elif _modo_voz == "powershell":
        def trecho(t, tm, rt):
            porcento = round(((1 + rt / 10) * fator - 1) * 100)
            return f'<prosody pitch="{tm * 5:+d}%" rate="{porcento:+d}%">{escape(t)}</prosody>'
        ssml = ('<speak version="1.0" xmlns="http://www.w3.org/2001/10/synthesis" xml:lang="pt-BR">'
                + (trecho(nome + ".", tom_n, ritmo_n) + " " if nome else "")
                + trecho(texto, tom, ritmo) + '</speak>')
        _voz.stdin.write(base64.b64encode(ssml.encode("utf-8")).decode("ascii") + "\n")
        _voz.stdin.flush()
        _voz.stdout.readline()          # espera o "OK" = terminou de falar
    elif _modo_voz == "pyttsx3":
        # o pyttsx3 só fala a 1ª frase se reaproveitar o motor: cria um novo a cada fala
        motor = _voz.init()
        motor.setProperty("rate", int(180 * fator))
        motor.say((nome + ". " if nome else "") + texto)
        motor.runAndWait()
        motor.stop()
        del motor


def achar_audio(nome, pasta=None):
    """Caminho da gravação 'nome' (sem extensão) ou None se ainda não foi gravada."""
    pasta = pasta or PASTA_AUDIOS
    for ext in EXTENSOES:
        caminho = os.path.join(pasta, nome + ext)
        if os.path.exists(caminho):
            return caminho
    return None


def tocar(caminho):
    """Toca um arquivo de áudio até o fim (ou até o jogador apertar uma tecla)."""
    if pygame is not None:
        pygame.mixer.music.load(caminho)
        pygame.mixer.music.play()

        def terminou():
            time.sleep(0.03)
            return not pygame.mixer.music.get_busy()
        acompanhar(terminou, pygame.mixer.music.stop)
    elif winsound is not None:
        winsound.PlaySound(caminho, winsound.SND_FILENAME)


def tocar_gravacoes(nomes, pasta=None):
    """Toca as gravações em sequência. Se faltar alguma, não toca nada e retorna False."""
    caminhos = [achar_audio(n, pasta) for n in nomes]
    if not nomes or None in caminhos or (pygame is None and winsound is None):
        return False
    for c in caminhos:
        tocar(c)
        if tecla_antecipada is not None:   # o jogador interrompeu: não toca o resto
            break
    return True


def esperar(segundos):
    """Pausa entre falas: encurta junto com a velocidade da fala escolhida no menu."""
    if VELOCIDADE > 0:
        time.sleep(segundos * VELOCIDADE / globals().get("velocidade_da_fala", 1.0))


iniciar_voz()

## 1.3 Personagens, falas e efeitos sonoros
`FALAS` guarda todas as falas do jogo: `código -> (personagem, texto)`.
Cada cena registra as suas falas no próprio bloco. **O código da fala é o nome do arquivo de áudio** que o integrante grava.
- `dizer("A02")` toca `audios/A02.wav`.
- `anunciar(quem, texto, pecas=[...])` monta uma frase a partir de vários pedaços gravados (usado para ler matrizes sorteadas).

**Voz sintetizada por personagem:** enquanto não há gravação, cada personagem fala com a voz da Maria num **tom e ritmo próprios** (tabela `VOZES_PERSONAGENS`): o Alquimista mais grave, o Grilo agudo e rápido, o Goblin agudo e apressado, o Max grave e arrastado.
Quando o personagem muda, a voz diz o nome dele antes da fala ("Alquimista. Ah, é você..."). O **R** sempre repete com o nome.

In [ ]:
NARRADOR = "Narrador"
SISTEMA = "Sistema"
ALQUIMISTA = "Alquimista"
GRILO = "Grilo Falante"
GOBLIN = "Goblin"
MAX = "Max"
VOZ_MISTERIOSA = "Voz misteriosa"
CAVALEIRO = "Cavaleiro Fantasma"
GOLEM = "Golem"
DRAGAO = "Dragão"

# Voz de cada personagem na voz sintetizada: (tom, ritmo) de -10 a 10 com a voz feminina (Maria),
# ou (tom, ritmo, "masculina", tom de reserva) com a voz masculina (Daniel). O tom de reserva é usado
# na Maria se o computador não tiver o Daniel.
VOZES_PERSONAGENS = {
    NARRADOR: (0, 0),
    SISTEMA: (3, 1),
    GRILO: (9, 2),
    VOZ_MISTERIOSA: (-10, -4),
    ALQUIMISTA: (0, -1, "masculina", -4),
    GOBLIN: (8, 3, "masculina", 6),
    MAX: (-5, -3, "masculina", -8),
    CAVALEIRO: (-7, -2, "masculina", -6),
    GOLEM: (-10, -3, "masculina", -9),
    DRAGAO: (-10, -1, "masculina", -10),
}

FALAS = {}
ultima_fala = (NARRADOR, "", None)
_ultimo_falante = None
em_pergunta = False   # True enquanto o jogo lê uma pergunta e as opções dela


from contextlib import contextmanager


@contextmanager
def pergunta():
    """Marca a leitura de uma pergunta e das opções: se o jogador apertar uma tecla
    no meio, o jogo para de ler o resto e usa a tecla como resposta."""
    global em_pergunta
    anterior, em_pergunta = em_pergunta, True
    try:
        yield
    finally:
        em_pergunta = anterior


TECLAS_VELOCIDADE = {"+": +1, "=": +1, "-": -1, "_": -1}   # + (ou =) acelera; - desacelera


def ajustar_velocidade(direcao, anunciar_mudanca=True):
    """Sobe (+1) ou desce (-1) um degrau em VELOCIDADES_DA_FALA. Retorna True se mudou.
    - No menu (anunciar_mudanca=True): fala a velocidade nova, ou que chegou no limite.
    - Durante o jogo (False): não fala nada, para não atrapalhar a história."""
    global velocidade_da_fala
    lista = VELOCIDADES_DA_FALA
    atual = lista.index(velocidade_da_fala) if velocidade_da_fala in lista else lista.index(1.0)
    novo = atual + direcao
    if not 0 <= novo < len(lista):
        if anunciar_mudanca:
            dizer("N08_max" if direcao > 0 else "N08_min")
        return False
    velocidade_da_fala = lista[novo]
    if anunciar_mudanca:
        dizer(f"N08_{velocidade_da_fala}")
    return True


COMANDOS_NA_FALA = {"R", "P", "V", "H", "O", "Q", "M"}


def _comando_durante_a_fala():
    """O jogador apertou R, P, V, H, O ou Q no meio de uma fala da história?
    Faz o comando na hora e retorna True para a fala interrompida recomeçar.
    (Numa pergunta, quem trata essas teclas é o escolher(), e a fala não recomeça.)"""
    global tecla_antecipada
    tecla = tecla_antecipada
    if tecla not in COMANDOS_NA_FALA or em_pergunta or (tecla == "M" and not em_jogo):
        return False
    tecla_antecipada = None
    if tecla == "M":
        raise VoltarAoMenu
    if tecla == "Q":
        raise SairDoJogo
    if tecla == "P":
        pausar()
    elif tecla == "V":
        dizer(f"S06_{estado.vidas}")
    elif tecla == "H":
        dizer("S12")
    return True    # R e O: só repetem a fala; P, V e H: repetem depois do comando


def _ja_respondeu():
    """O jogador apertou uma tecla durante a fala anterior?
    - Numa pergunta: sim, então não lê o resto (a tecla vai ser a resposta).
    - Fora dela: a tecla só serviu para pular aquela fala e é descartada."""
    global tecla_antecipada
    if tecla_antecipada is None:
        return False
    if em_pergunta:
        return True
    tecla_antecipada = None
    return False


def anunciar(quem, texto, pecas=None, pausa=0.6, dizer_nome=False):
    """Imprime o texto e toca as gravações em `pecas` (lista de códigos).
    Se faltar alguma gravação, usa a voz sintetizada no tom do personagem,
    dizendo o nome dele quando o personagem muda (ou quando `dizer_nome`)."""
    global ultima_fala, _ultimo_falante
    personagem = quem not in (NARRADOR, SISTEMA)
    print((f"{quem.upper()}: " if personagem else "") + texto)
    ultima_fala = (quem, texto, pecas)
    if _ja_respondeu():
        return
    while True:
        if not (usar_gravacoes and tocar_gravacoes(pecas or [])):
            nome = quem if (DIZER_NOME_DO_PERSONAGEM and personagem
                            and (dizer_nome or quem != _ultimo_falante)) else None
            falar_em_voz(texto, quem, nome)
        _ultimo_falante = quem
        if not _comando_durante_a_fala():
            break
        dizer_nome = True   # R, P, V, H ou O no meio da fala: ela recomeça do início
    if tecla_antecipada is None:
        esperar(pausa)


def dizer(codigo, pausa=0.6):
    """Fala a fala `codigo` de FALAS, tocando audios/<codigo>.wav."""
    quem, texto = FALAS[codigo]
    anunciar(quem, texto, [codigo], pausa)


def som(descricao, arquivo):
    """Efeito sonoro (audios/sons/<arquivo>.wav). Sem a gravação, o efeito é
    descrito em voz alta, porque o jogador cego depende dele."""
    print(f"[som: {descricao}]")
    if _ja_respondeu():
        return
    while True:
        if not tocar_gravacoes([arquivo], os.path.join(PASTA_AUDIOS, "sons")):
            falar_em_voz(f"Som: {descricao}")
        if not _comando_durante_a_fala():
            break
    if tecla_antecipada is None:
        esperar(0.4)

## 1.4 Comandos do jogador (ASDW)
`escolher()` é o único ponto onde o jogo lê o teclado. Ele já trata **R** (repetir), **P** (pausar), **V** (vidas), **H** (ajuda) e **Q** (sair) em qualquer momento.
Quando a fala termina e o jogo passa a esperar uma tecla, toca um **sinal sonoro** (`audios/sons/vez_do_jogador.wav` ou, sem ele, um bipe curto). Ele é configurado em `SINAL_DE_ESPERA`.

**Tecla O (opções de agora):** fala só as teclas que valem naquele momento, com o que cada uma faz (ex.: "A para o caminho da floresta; D para o caminho vermelho"). O **H** continua listando todos os comandos. Como a lista é montada na hora, ela sai sempre na voz sintetizada.

**Responder sem esperar a fala acabar** (no `.exe` e no terminal):
- A tecla funciona **na hora, sem Enter**, e as **setas** do teclado valem como W, A, S, D.
- Apertou uma tecla **durante uma fala**? A voz para na hora.
  - Se for uma **pergunta** (o menu, as escolhas, as 3 opções de matriz), a tecla **já vale como resposta**, e o jogo não lê o resto das opções.
  - Se for uma fala da história, a tecla **pula aquela fala**. As exceções são os comandos: **R** recomeça a fala; **P** pausa, e **V** e **H** dizem as vidas e os comandos, e depois a fala recomeça; **Q** sai do jogo.
- Funciona com a voz da Maria (SAPI) e com as gravações (pygame). Pode ser desligado em `PODE_INTERROMPER`.

**Velocidade da fala a qualquer momento:** **+** (ou **=**) acelera e **−** desacelera, de 0,75x a 3x. Durante o jogo, **nada é anunciado**, para não atrapalhar a história:
- No meio de uma fala, a voz **acelera ou desacelera na hora, sem parar nem recomeçar** (com a voz da Maria; as gravações continuam em 1x).
- Esperando uma resposta, toca só um bipe curto: agudo se acelerou, grave se desacelerou. Desligue em `BIPE_DE_VELOCIDADE`. No limite (0,75x ou 3x), nada acontece.
- O anúncio falado ("Velocidade da fala: uma vez e meia") fica só no ajuste do menu (tecla W).
- No Jupyter, digite + ou − e aperte Enter.
- **No Jupyter** isso não funciona (lá não dá para ler o teclado na hora): é preciso esperar a fala, digitar a tecla e apertar **Enter**.

In [ ]:
class SairDoJogo(Exception):
    pass


class Derrota(Exception):
    pass


class VoltarAoMenu(Exception):
    pass


em_jogo = False   # True enquanto uma partida está rodando (a tecla M só vale nela)


def sinal_de_espera():
    """Som curto avisando que a fala acabou e o jogo espera uma tecla.
    Usa audios/sons/vez_do_jogador.wav se existir; senão, um bipe do Windows."""
    if not SINAL_DE_ESPERA:
        return
    if not tocar_gravacoes(["vez_do_jogador"], os.path.join(PASTA_AUDIOS, "sons")) and winsound:
        winsound.Beep(880, 120)


def ler_tecla():
    """Lê a tecla do jogador.
    - Se ele já apertou durante a fala, usa essa tecla.
    - No .exe/terminal: lê na hora, sem Enter (setas também valem).
    - No Jupyter: digita a tecla e aperta Enter."""
    global tecla_antecipada
    if tecla_antecipada is not None:
        tecla, tecla_antecipada = tecla_antecipada, None
        print(f"> {tecla}")
        return tecla
    if MODO_TESTE:
        if not ENTRADAS_TESTE:
            raise SairDoJogo
        tecla = ENTRADAS_TESTE.pop(0)
        print(f"> {tecla}")
    elif console_interativo():
        sinal_de_espera()
        print("> ", end="", flush=True)
        tecla = ler_tecla_console()
        print(tecla)
        return tecla
    else:
        sinal_de_espera()
        tecla = input("> ")
    return tecla.strip().upper()[:1]


def repetir(bloco):
    """Repete o bloco (ex.: pergunta + opções) ou, se não houver, a última fala.
    Dá para responder no meio da repetição."""
    with pergunta():
        for quem, texto, pecas in (bloco or [ultima_fala]):
            anunciar(quem, texto, pecas, pausa=0.2, dizer_nome=True)


def pausar():
    global tecla_antecipada
    tecla_antecipada = None
    dizer("S03")
    tecla_antecipada = None
    if console_interativo():
        ler_tecla_console()
    elif not MODO_TESTE:
        input()
    dizer("S04")


def falar_opcoes(opcoes):
    """Tecla O: fala só as teclas que valem agora (ex.: "A para o caminho da floresta").
    É montada na hora, então sai sempre na voz sintetizada."""
    global ultima_fala
    guardada = ultima_fala               # o R continua repetindo a fala da história, não esta lista
    partes = [f"{t} para {d}" for t, d in opcoes.items()]
    anunciar(SISTEMA, "Agora você pode apertar: " + "; ".join(partes) + ". E H para todos os comandos.")
    ultima_fala = guardada


def escolher(opcoes, certa=None, bloco_repetir=None, aceita_enter=False):
    """Espera uma tecla válida de `opcoes` (dict tecla -> descrição).
    `certa` marca a resposta esperada (usada nos testes automáticos).
    `bloco_repetir` é a lista de falas que o R repete (ex.: pergunta + opções).
    Se não vier bloco, o R repete a fala dita logo antes da pergunta."""
    estado.resposta_certa = certa or next(iter(opcoes))
    bloco = bloco_repetir or [ultima_fala]
    while True:
        tecla = ler_tecla()
        if tecla in opcoes:
            return tecla
        if tecla == "" and aceita_enter:
            return next(iter(opcoes))
        if tecla in TECLAS_VELOCIDADE:         # muda em silêncio; só um bipe curto confirma
            direcao = TECLAS_VELOCIDADE[tecla]
            if ajustar_velocidade(direcao, anunciar_mudanca=False) and BIPE_DE_VELOCIDADE and winsound:
                winsound.Beep(1046 if direcao > 0 else 523, 70)
        elif tecla == "R":
            repetir(bloco)
        elif tecla == "P":
            pausar()
        elif tecla == "V":
            dizer(f"S06_{estado.vidas}")
        elif tecla == "H":
            dizer("S12")
        elif tecla == "O":
            falar_opcoes(opcoes)
        elif tecla == "M" and em_jogo:
            raise VoltarAoMenu
        elif tecla == "Q":
            raise SairDoJogo
        else:
            dizer("S01")        # tecla errada: avisa e repete as opções
            repetir(bloco)


_avisos_continuar = 0


def continuar():
    """Pausa a história até o jogador apertar W (ou só Enter)."""
    global _avisos_continuar
    if _avisos_continuar < 3:          # depois de 3 avisos, o jogador já aprendeu
        dizer("S09", pausa=0)
        _avisos_continuar += 1
    else:
        print("[W] continuar")
    escolher({"W": "continuar"}, aceita_enter=True)

## 1.5 Estado do jogador: vidas, magias e checkpoint

In [ ]:
VIDAS_INICIAIS = 3


class Estado:
    def __init__(self):
        self.vidas = VIDAS_INICIAIS
        self.magias = []
        self.checkpoint = None     # cena para onde o jogador volta ao perder todas as vidas
        self.retorno = False       # True quando acabou de voltar de uma derrota
        self.resposta_certa = None
        self.modo = "demo"          # "demo" ou "completo"
        self.caminho = None        # na masmorra: "cavaleiro" ou "golem"


estado = Estado()


def perder_vida():
    estado.vidas -= 1
    som("um impacto seco", "dano")
    if estado.vidas > 0:
        dizer(f"S02_{estado.vidas}")      # S02_2 ou S02_1
    else:
        dizer("S05")
        raise Derrota


def aprender_magia(codigo_magia):
    """codigo_magia: 'soma', 'triangular' ou 'nula' (fala S11_<codigo>)."""
    if codigo_magia not in estado.magias:
        estado.magias.append(codigo_magia)
    som("a pedra pulsa com um brilho quente", "magia")
    dizer(f"S11_{codigo_magia}")

## 1.6 Matemática: matrizes faladas, desenhadas e classificadas
O jogador não vê a matriz, então ela é **lida em voz alta** (`ler_matriz`) e, para quem enxerga, também **desenhada** (`desenhar_matriz`).
Como as matrizes da tempestade são sorteadas, a leitura é **montada com pedaços gravados** (`NUM_3`, `MAT_EM_CIMA`, `MAT_E`...). Os textos desses pedaços estão na Parte 2.
Toda classificação vem com **justificativa falada** (pedido da professora).

In [ ]:
TECLAS_OPCOES = ["A", "W", "D"]   # esquerda, meio, direita


class Frase:
    """Frase montada de pedaços gravados: guarda o texto e a lista de códigos de áudio."""
    def __init__(self):
        self.textos, self.pecas = [], []

    def add(self, codigo):
        self.textos.append(FALAS[codigo][1])
        self.pecas.append(codigo)
        return self

    def num(self, n):
        n = int(n)
        if n < 0:
            self.add("MENOS")
        return self.add(f"NUM_{abs(n)}")

    @property
    def texto(self):
        return " ".join(self.textos)


def ler_matriz(M, frase=None):
    """Monta a leitura da matriz com pedaços gravados:
    2x2 -> 'em cima, dois e um, embaixo, zero e três'
    3x3 -> 'primeira linha, ..., segunda linha, ..., terceira linha, ...'"""
    M = np.array(M)
    frase = frase or Frase()
    for i, linha in enumerate(M):
        if M.shape[0] == 2:
            frase.add("MAT_EM_CIMA" if i == 0 else "MAT_EMBAIXO")
        else:
            frase.add(f"MAT_LINHA_{i + 1}")
        for j, x in enumerate(linha):
            if j == len(linha) - 1 and j > 0:
                frase.add("MAT_E")
            frase.num(x)
            if j < len(linha) - 2:
                frase.textos[-1] += ","          # pontuação só no texto impresso
        frase.textos[-1] += "." if i == len(M) - 1 else ";"
    return frase


def desenhar_matriz(M):
    M = np.array(M)
    largura = max(len(str(x)) for x in M.flat)
    return "\n".join("      | " + "  ".join(str(x).rjust(largura) for x in linha) + " |" for linha in M)


def e_triangular_superior(M):
    M = np.array(M)
    return M.shape[0] == M.shape[1] and bool(np.all(np.tril(M, -1) == 0))


def justificar_triangular(M):
    """Justificativa falada da classificação (Frase com pedaços gravados J01..J06)."""
    M = np.array(M)
    n = M.shape[0]
    frase = Frase()
    abaixo = [(i, j, M[i, j]) for i in range(n) for j in range(i) if M[i, j] != 0]
    if not abaixo:
        return frase.add("J01")
    i, j, v = abaixo[0]
    frase.add("J02").num(i + 1).add("J03").num(j + 1).add("J04").num(v).add("J05")
    if np.all(np.triu(M, 1) == 0):
        frase.add("J06")
    return frase


def matriz_aleatoria(n):
    return np.array([[random.randint(1, 9) for _ in range(n)] for _ in range(n)])


def perguntar_opcoes(codigo_pergunta, frases, indice_certo, matrizes=None):
    """Fala a pergunta (código de FALAS) e as 3 opções (A, W, D), cada uma uma Frase.
    Se vierem `matrizes`, também desenha cada uma na tela. Retorna o índice escolhido."""
    with pergunta():   # apertou A, W ou D no meio da leitura? O jogo para de ler e já responde
        dizer(codigo_pergunta)
        bloco = [ultima_fala]
        for k, frase in enumerate(frases):
            anunciar(NARRADOR, frase.texto, frase.pecas, pausa=0.3)
            bloco.append(ultima_fala)
            if matrizes is not None:
                print(desenhar_matriz(matrizes[k]))
    opcoes = {t: f"a opção {t}" for t in TECLAS_OPCOES}
    tecla = escolher(opcoes, certa=TECLAS_OPCOES[indice_certo], bloco_repetir=bloco)
    return TECLAS_OPCOES.index(tecla)


def perguntar_matrizes(codigo_pergunta, matrizes, indice_certo):
    """Pergunta em que cada opção é uma matriz, lida em voz alta."""
    frases = [ler_matriz(M, Frase().add(f"OPCAO_{t}")) for t, M in zip(TECLAS_OPCOES, matrizes)]
    return perguntar_opcoes(codigo_pergunta, frases, indice_certo, matrizes)


# ---------------------------------------------------------------- tipos de matriz
# Os tipos se sobrepõem (a identidade também é diagonal, triangular e simétrica; a nula é tudo isso).
# Por isso o jogo pede sempre o tipo MAIS ESPECÍFICO, nesta ordem:
ORDEM_DOS_TIPOS = ["nula", "identidade", "diagonal", "triangular", "simetrica"]


def e_nula(M):
    return bool(np.all(np.array(M) == 0))


def e_quadrada(M):
    M = np.array(M)
    return M.ndim == 2 and M.shape[0] == M.shape[1]


def e_diagonal(M):
    M = np.array(M)
    return e_quadrada(M) and bool(np.all(M == np.diag(np.diag(M))))


def e_identidade(M):
    M = np.array(M)
    return e_quadrada(M) and bool(np.array_equal(M, np.eye(len(M), dtype=int)))


def e_simetrica(M):
    M = np.array(M)
    return e_quadrada(M) and bool(np.array_equal(M, M.T))


E_DO_TIPO = {
    "nula": e_nula,
    "identidade": e_identidade,
    "diagonal": e_diagonal,
    "triangular": e_triangular_superior,
    "simetrica": e_simetrica,
}


def tipo_mais_especifico(M):
    """O primeiro tipo de ORDEM_DOS_TIPOS em que a matriz se encaixa (ou None)."""
    return next((t for t in ORDEM_DOS_TIPOS if E_DO_TIPO[t](M)), None)


def gerar_matriz_do_tipo(tipo, n=2):
    """Matriz n x n cujo tipo mais específico é exatamente `tipo` ("cheia" = nenhum tipo)."""
    while True:
        if tipo == "nula":
            M = np.zeros((n, n), dtype=int)
        elif tipo == "identidade":
            M = np.eye(n, dtype=int)
        elif tipo == "diagonal":
            M = np.diag([random.randint(2, 9) for _ in range(n)])
        elif tipo == "triangular":
            M = np.triu(matriz_aleatoria(n))
        elif tipo == "simetrica":
            A = np.triu(matriz_aleatoria(n))
            M = A + np.triu(A, 1).T
        else:  # "cheia": todos os números diferentes de zero e não simétrica
            M = matriz_aleatoria(n)
        if tipo_mais_especifico(M) == (None if tipo == "cheia" else tipo):
            return M


def justificar_tipo(M, tipo_pedido=None):
    """Justificativa falada do tipo da matriz.
    Se `tipo_pedido` vier e não for o tipo dela, explica por quê (inclusive quando ela até se
    encaixa no tipo pedido, mas existe um tipo mais específico)."""
    real = tipo_mais_especifico(M)
    frase = Frase()
    if tipo_pedido and tipo_pedido != real:
        frase.add(f"JN_{tipo_pedido}")              # "Não é a matriz X."
        if E_DO_TIPO[tipo_pedido](M):
            frase.add("J_MAIS_ESPECIFICO")          # "Ela até é, mas o tipo mais específico é outro."
    return frase.add(f"JT_{real}" if real else "JT_cheia")


def opcoes_de_tipo(certo, conhecidos):
    """3 tipos para uma pergunta de classificação: o certo + 2 outros dos que o jogador conhece."""
    outros = [t for t in conhecidos if t != certo]
    random.shuffle(outros)
    opcoes = [certo] + outros[:2]
    random.shuffle(opcoes)
    return opcoes


def desafio_matrizes(codigo_pergunta, opcoes, prefixo_erro, fala_erro=None):
    """Pergunta de 3 opções. `opcoes` = lista de (nome, matriz); a de nome "certa" é a resposta.
    Errou: fala `fala_erro` (se houver) e `<prefixo_erro>_<nome da opção>`, perde 1 vida e
    a pergunta volta com as opções embaralhadas."""
    while True:
        random.shuffle(opcoes)
        certo = next(i for i, (nome, _) in enumerate(opcoes) if nome == "certa")
        escolha = perguntar_matrizes(codigo_pergunta, [M for _, M in opcoes], certo)
        if escolha == certo:
            dizer("S07")
            return
        if fala_erro:
            dizer(fala_erro)
        dizer(f"{prefixo_erro}_{opcoes[escolha][0]}")
        perder_vida()

---
# Parte 2 — Falas do Narrador e do sistema
Tudo o que o jogo diz fora da história: nome do jogo, menu (**DEMO** ou **jogo completo**), instruções, comandos, vidas e pausa.

In [ ]:
FALAS.update({
    # --- Abertura e menu ---
    "N01": (NARRADOR, "Mathrix Souls."),
    "N02": (NARRADOR, "Um RPG de fantasia sombria em que as matrizes são a sua magia. "
                      "Feito para ser jogado de olhos fechados e ouvidos atentos."),
    "N03": (NARRADOR, "Menu principal. Aperte A para jogar a DEMO. Aperte D para o jogo completo. "
                      "Aperte C para escolher uma cena. Aperte S para ouvir como jogar. "
                      "Aperte G para escolher entre vozes gravadas e vozes do computador. "
                      "Aperte H para ouvir todos os comandos. Aperte Q para sair."),
    "N04": (NARRADOR, "Iniciando o jogo completo: do despertar do jovem mago até o dragão "
                      "que dorme sob o farol."),
    "N05": (NARRADOR, "Como jogar. Você vai ouvir a história e responder com as teclas W, A, S e D, "
                      "ou com as setas do teclado. A é esquerda, D é direita, W é frente ou confirmar, "
                      "S é trás ou recuar. Nas perguntas com três respostas, A é a primeira, "
                      "W é a do meio e D é a última. A qualquer momento: R repete a última fala, "
                      "P pausa, V diz suas vidas, O diz as opções de agora, H lista os comandos, M volta ao menu principal salvando o jogo, e Q sai do jogo. "
                      "Não precisa esperar a fala terminar: aperte a tecla quando quiser, e a "
                      "voz para. As teclas mais e menos aceleram ou desaceleram a fala, até três vezes, a qualquer "
                      "momento. No menu, a tecla C escolhe a cena para começar. Você tem três vidas."),
    "N06": (NARRADOR, "Iniciando a DEMO: o encontro com o alquimista, o combate com o goblin "
                      "e a travessia de barco até o farol."),
    "N07": (NARRADOR, "Até a próxima, jovem mago."),
    "N09": (NARRADOR, "Velocidade da fala. Aperte D para aumentar, A para diminuir, e W para voltar ao menu. "
                      "Durante o jogo, as teclas mais e menos fazem o mesmo."),
    "N08_0.75": (NARRADOR, "Velocidade da fala: mais lenta."),
    "N08_1.0": (NARRADOR, "Velocidade da fala: normal."),
    "N08_1.25": (NARRADOR, "Velocidade da fala: uma vez e um quarto."),
    "N08_1.5": (NARRADOR, "Velocidade da fala: uma vez e meia."),
    "N08_2.0": (NARRADOR, "Velocidade da fala: duas vezes."),
    "N08_2.5": (NARRADOR, "Velocidade da fala: duas vezes e meia."),
    "N08_3.0": (NARRADOR, "Velocidade da fala: três vezes. Essa é a mais rápida."),
    "N08_max": (NARRADOR, "Essa já é a velocidade mais rápida."),
    # --- Seleção de cenas ---
    "N160": (NARRADOR, "Vozes gravadas pela equipe. Se faltar alguma gravação, a voz do computador completa."),
    "N161": (NARRADOR, "Vozes do computador."),
    # --- Salvamento ---
    "SV01": (SISTEMA, "Você tem um jogo salvo. Para continuar de onde parou, aperte E."),
    "SV02": (SISTEMA, "Continuando de onde você parou."),
    "SV03": (SISTEMA, "Antes de sair, quer salvar o jogo? Aperte W para salvar e sair, S para sair sem salvar, ou A para voltar ao jogo. A cena recomeça do início."),
    "SV04": (SISTEMA, "Jogo salvo. Quando voltar, você poderá continuar daqui."),
    "SV07": (SISTEMA, "Voltando ao jogo."),
    "SV06": (SISTEMA, "Jogo salvo. Voltando ao menu principal."),
    "SV05": (SISTEMA, "Não consegui salvar o jogo neste computador."),
    "N140": (NARRADOR, "Seleção de cenas. Aperte S para a próxima cena, W para a anterior, D para "
                       "jogar a cena escolhida, e A para voltar ao menu. O jogo segue dali até o fim."),
    "N141": (NARRADOR, "Começando pela cena escolhida."),
    "SC_1": (NARRADOR, "Cena 1: a casa."),
    "SC_2": (NARRADOR, "Cena 2: a taverna."),
    "SC_3": (NARRADOR, "Cena 3: a lição da matriz quadrada."),
    "SC_4": (NARRADOR, "Cena 4: o grilo falante."),
    "SC_5": (NARRADOR, "Cena 5: a bifurcação."),
    "SC_6": (NARRADOR, "Cena 6: o goblin."),
    "SC_7": (NARRADOR, "Cena 7: a praia."),
    "SC_8": (NARRADOR, "Cena 8: o barco e a tempestade."),
    "SC_9": (NARRADOR, "Cena 9: a ilha e o farol."),
    "SC_10": (NARRADOR, "Cena 10: a masmorra."),
    "SC_11A": (NARRADOR, "Cena 11 A: o cavaleiro fantasma e o banquete."),
    "SC_11B": (NARRADOR, "Cena 11 B: o golem e o puzzle das pedras."),
    "SC_12": (NARRADOR, "Cena 12: o dragão."),
    "SC_EPI": (NARRADOR, "Epílogo: de volta à taverna."),
    "SC_FIM": (NARRADOR, "Essa é a última cena."),
    "SC_INI": (NARRADOR, "Essa é a primeira cena."),
    "N08_min": (NARRADOR, "Essa já é a velocidade mais lenta."),

    # --- Sistema ---
    "S01": (SISTEMA, "Essa tecla não vale aqui."),
    "S02_2": (SISTEMA, "Você perdeu uma vida. Restam duas vidas."),
    "S02_1": (SISTEMA, "Você perdeu uma vida. Resta só uma vida. Cuidado!"),
    "S03": (SISTEMA, "Jogo pausado. Aperte Enter para continuar."),
    "S04": (SISTEMA, "Jogo retomado."),
    "S05": (SISTEMA, "Suas vidas acabaram. Tudo escurece..."),
    "S06_3": (SISTEMA, "Você tem três vidas."),
    "S06_2": (SISTEMA, "Você tem duas vidas."),
    "S06_1": (SISTEMA, "Você tem uma vida."),
    "S07": (SISTEMA, "Acertou!"),
    "S09": (SISTEMA, "Aperte W para continuar."),
    "S10": (SISTEMA, "Esquivou!"),
    "S11_soma": (SISTEMA, "Nova magia aprendida: Matriz Quadrada. Agora você sabe somar matrizes."),
    "S11_triangular": (SISTEMA, "Nova magia aprendida: Matriz Triangular superior."),
    "S11_nula": (SISTEMA, "Nova magia aprendida: Matriz Nula. Todos os números dela são zero."),
    "S11_diagonal": (SISTEMA, "Nova magia aprendida: Matriz Diagonal."),
    "S11_identidade": (SISTEMA, "Nova magia aprendida: Matriz Identidade."),
    "S11_transposta": (SISTEMA, "Nova magia aprendida: Matriz Transposta. As linhas viram colunas."),
    "S11_simetrica": (SISTEMA, "Nova magia aprendida: Matriz Simétrica. Ela é igual à sua transposta."),
    "S12": (SISTEMA, "Comandos. W: frente, confirmar ou a opção do meio. A: esquerda ou a primeira opção. "
                     "S: trás ou recuar. D: direita ou a terceira opção. R: repetir a última fala. "
                     "P: pausar. V: ouvir suas vidas. O: ouvir só as opções de agora. H: ouvir os comandos. "
                     "M: voltar ao menu principal, salvando o jogo. Q: sair do jogo. "
                     "Tecla mais: acelerar a fala. Tecla menos: desacelerar a fala. As setas do teclado também "
                     "valem. Aperte a tecla quando quiser, mesmo no meio de uma fala."),
})

### Pedaços para montar frases (leitura de matrizes e justificativas)
As matrizes da tempestade, das pedras e do dragão são sorteadas, então não dá para gravar a frase inteira. O jogo junta estes pedaços.
Exemplo: `OPCAO_A` + `MAT_EM_CIMA` + `NUM_2` + `MAT_E` + `NUM_1` + `MAT_EMBAIXO` + `NUM_0` + `MAT_E` + `NUM_3`
soa como: *"Opção A: em cima, dois e um; embaixo, zero e três."*
**Grave cada pedaço curto, sem silêncio no começo nem no fim.**

In [ ]:
EXTENSO = ["zero", "um", "dois", "três", "quatro", "cinco", "seis", "sete", "oito", "nove", "dez",
           "onze", "doze", "treze", "quatorze", "quinze", "dezesseis", "dezessete", "dezoito",
           "dezenove", "vinte"]

FALAS.update({f"NUM_{n}": (NARRADOR, nome) for n, nome in enumerate(EXTENSO)})
FALAS.update({
    "MENOS": (NARRADOR, "menos"),
    "MAT_E": (NARRADOR, "e"),
    "MAT_EM_CIMA": (NARRADOR, "em cima,"),
    "MAT_EMBAIXO": (NARRADOR, "embaixo,"),
    "MAT_LINHA_1": (NARRADOR, "primeira linha,"),
    "MAT_LINHA_2": (NARRADOR, "segunda linha,"),
    "MAT_LINHA_3": (NARRADOR, "terceira linha,"),
    "OPCAO_A": (NARRADOR, "Opção A:"),
    "OPCAO_W": (NARRADOR, "Opção W:"),
    "OPCAO_D": (NARRADOR, "Opção D:"),
    # justificativas da triangular superior
    "J01": (NARRADOR, "É triangular superior porque todos os números abaixo da diagonal principal são zero."),
    "J02": (NARRADOR, "Não é triangular superior. Abaixo da diagonal, na linha"),
    "J03": (NARRADOR, "coluna"),
    "J04": (NARRADOR, "tem o número"),
    "J05": (NARRADOR, "e ali deveria ser zero."),
    "J06": (NARRADOR, "Essa é a triangular inferior: os zeros estão acima da diagonal, não abaixo."),
    # tipos de matriz (opções faladas)
    "TIPO_nula": (NARRADOR, "matriz nula."),
    "TIPO_identidade": (NARRADOR, "matriz identidade."),
    "TIPO_diagonal": (NARRADOR, "matriz diagonal."),
    "TIPO_triangular": (NARRADOR, "triangular superior."),
    "TIPO_simetrica": (NARRADOR, "matriz simétrica."),
    # justificativas por tipo (o tipo mais específico)
    "JT_nula": (NARRADOR, "É a matriz nula: todos os números são zero."),
    "JT_identidade": (NARRADOR, "É a matriz identidade: um em toda a diagonal principal e zero no resto."),
    "JT_diagonal": (NARRADOR, "É uma matriz diagonal: fora da diagonal principal, todos os números são zero."),
    "JT_triangular": (NARRADOR, "É triangular superior: abaixo da diagonal é tudo zero, mas acima tem números."),
    "JT_simetrica": (NARRADOR, "É simétrica: os números se espelham na diagonal principal, então ela é "
                               "igual à sua transposta."),
    "JT_cheia": (NARRADOR, "Não é nenhum desses tipos: tem números fora da diagonal, dos dois lados, e eles "
                           "não se espelham."),
    "JN_nula": (NARRADOR, "Não é a matriz nula."),
    "JN_identidade": (NARRADOR, "Não é a matriz identidade."),
    "JN_diagonal": (NARRADOR, "Não é a matriz diagonal."),
    "JN_triangular": (NARRADOR, "Não é a triangular superior."),
    "JN_simetrica": (NARRADOR, "Não é a matriz simétrica."),
    "J_MAIS_ESPECIFICO": (NARRADOR, "Ela até se encaixa nesse tipo, mas existe um tipo mais específico."),
})

---
# Parte 3 — Cenas da DEMO
## Cena 1 — A Casa
**Vozes:** Narrador. Apresenta a motivação do herói: a **mãe doente** na casa de cura, aos cuidados do Druida, e as ervas caras que podem curá-la.

In [ ]:
FALAS.update({
    "N10": (NARRADOR, "Você é um jovem mago iniciante e acaba de acordar em uma pequena casa, "
                      "longe do vilarejo de Trim, em um continente desconhecido que guarda "
                      "propriedades misteriosas."),
    "N11": (NARRADOR, "Você teve um sonho: enfrentava grandes monstros e desafios com magias "
                      "absurdamente poderosas. Ao despertar, porém, lembra que ainda sabe pouco de magia "
                      "e não faz ideia de como usá-la a seu favor."),
    "N14": (NARRADOR, "O que realmente tira seu sono é a condição de sua mãe. Ela está doente e permanece na casa de cura do vilarejo"
                      ", sob os cuidados do Druida, mestre no preparo de poções e remédios. Só que as ervas raras capazes de salvá-la "
                      "custam mais ouro do que você jamais viu."),
    "N12": (NARRADOR, "Aperte W para pegar o seu antigo cajado de madeira e sair de casa."),
    "N13": (NARRADOR, "Entre a decepção e a preocupação, você pega o cajado e sai rumo à taverna, "
                      "atrás de um café da manhã e, quem sabe, de algum trabalho que pague bem."),
})

In [ ]:
def cena_1_casa():
    dizer("N10")
    som("um galo canta ao longe; o vento arranha a janela", "galo_vento")
    dizer("N11")
    dizer("N14")
    dizer("N12")
    escolher({"W": "pegar o cajado e sair"}, aceita_enter=True)
    som("porta de madeira rangendo", "porta")
    dizer("N13")

## Cena 2 — A Taverna
**Vozes:** Narrador, Max, Alquimista. O Alquimista oferece o **tesouro do Dragão** para pagar as ervas da mãe do herói: é o motivo para aceitar a missão.

In [ ]:
FALAS.update({
    "N20": (NARRADOR, "No balcão está seu velho amigo Max, passando mal: geme, suspira e deixa "
                      "a cabeça cair sobre a madeira, completamente bêbado."),
    "N21": (NARRADOR, "Aperte W para tentar falar com o Max, ou S para deixá-lo em paz."),
    "M01": (MAX, "Hmm... matriz... dragão... mais uma caneca, por favor..."),
    "N22": (NARRADOR, "Conhecendo o Max, você sabe que não arrancaria uma frase com sentido dele."),
    "N23": (NARRADOR, "Atrás de você, uma voz:"),
    "A01": (ALQUIMISTA, "Finalmente desmaiou. Credo, esse aí fala demais."),
    "N24": (NARRADOR, "Você se vira. Frascos tilintam, e o cheiro de ervas queimadas "
                      "chega antes das palavras. É o Alquimista Algébrico."),
    "A02": (ALQUIMISTA, "Ah, olha só, o jovem mago iniciante. Max e o Druida me falaram de você. "
                        "Quero te propor um acordo."),
    "N25": (NARRADOR, "Aperte W para aceitar, ou S para recusar."),
    "N26": (NARRADOR, "Antes que você diga qualquer coisa, ele te interrompe:"),
    "A03": (ALQUIMISTA, "Ah, ah, nem precisa responder. Ninguém recusa nada vindo do GRANDE "
                        "alquimista da cidade."),
    "N27": (NARRADOR, "Um estalar de dedos, um sopro de vento, e de repente vocês estão sentados "
                      "à mesa da sala de estar dele, com o cheiro de pão quente no ar."),
    "A04": (ALQUIMISTA, "Gostou do teleporte? A maioria vomita na primeira vez. Bem, estamos aqui porque quero te fazer uma proposta: "
                        "eu te ensino magia, e você mata um dragão pra mim, na masmorra do farol."),
    "A05": (ALQUIMISTA, "É uma bela viagem, se você sobreviver a ela. Você atravessa a floresta, chega à grande praia, "
                        "pega um barco e alcança a ilha."),
    "A06": (ALQUIMISTA, "Eu iria sozinho, claro, mas o Druida me proibiu de sair do vilarejo. Neste período do "
                        "ano, as criaturas de fora ficam mais agitadas, e ele me pediu para ficar aqui e proteger "
                        "todo mundo. Sobre o dragão, não se preocupe: ele "
                        "tem um ponto fraco."),
    "A08": (ALQUIMISTA, "E não me olhe com essa cara: sei da sua mãe, e ela precisa dos remédios logo. Dragões dormem em cima de ouro, e o "
                        "tesouro dele é todo seu. Dá para pagar as ervas do Druida e ainda sobra."),
    "N28": (NARRADOR, "Ele te entrega uma pequena pedra. Ao tocar sua mão, ela é absorvida pela "
                      "pele, deixando um calor suave."),
    "A07": (ALQUIMISTA, "Fui eu que fiz. Ela libera o seu potencial mágico. Cada mago tem um tipo "
                        "de magia, e o seu são as matrizes. O tipo mais forte."),
})

In [ ]:
def cena_2_taverna():
    som("murmúrios, canecas batendo, lareira estalando", "taverna")
    dizer("N20")
    dizer("N21")
    if escolher({"W": "falar com o Max", "S": "deixar o Max em paz"}) == "W":
        dizer("M01")
    dizer("N22")
    dizer("N23")
    dizer("A01")
    som("frascos tilintando", "frascos")
    dizer("N24")
    dizer("A02")
    dizer("N25")
    escolher({"W": "aceitar", "S": "recusar"})   # tanto faz: ele interrompe!
    dizer("N26")
    dizer("A03")
    som("um estalar de dedos e um sopro de vento", "teleporte")
    dizer("N27")
    dizer("A04")
    dizer("A05")
    dizer("A06")
    dizer("A08")
    continuar()
    dizer("N28")
    dizer("A07")

## Cena 3 — A Lição da Matriz Quadrada
**Vozes:** Alquimista, Narrador. **Matemática:** matriz quadrada e soma (calculada com NumPy).

In [ ]:
FALAS.update({
    "A10": (ALQUIMISTA, "Uma matriz é uma tabela de números, com linhas e colunas. Quando tem o "
                        "mesmo número de linhas e colunas, é quadrada. Essa é a sua magia de poder normal."),
    "A11": (ALQUIMISTA, "Para somar duas matrizes, elas precisam ter o mesmo tamanho: o mesmo número "
                        "de linhas e de colunas. Aí você junta posição com posição. Veja: a primeira "
                        "tem um, dois na linha de cima, e três, quatro na de baixo. A segunda tem "
                        "cinco, seis em cima, e sete, oito embaixo."),
    "A12": (ALQUIMISTA, "Um mais cinco, seis. Dois mais seis, oito. Três mais sete, dez. "
                        "Quatro mais oito, doze. Simples, não? Eu sou um ótimo professor."),
    "N30": (NARRADOR, "Aperte W se entendeu, ou S para ouvir a explicação de novo."),
    "N31": (NARRADOR, "A pedra pulsa na sua pele. Você sente o poder da Matriz Quadrada passar a "
                      "fazer parte de você."),
})

In [ ]:
LICAO_A = np.array([[1, 2], [3, 4]])
LICAO_B = np.array([[5, 6], [7, 8]])


def cena_3_licao():
    som("o cajado bate no chão e ecoa como um sino grave", "cajado_sino")
    dizer("A10")
    while True:
        dizer("A11")
        print(desenhar_matriz(LICAO_A), "\n        +")
        print(desenhar_matriz(LICAO_B), "\n        =")
        print(desenhar_matriz(LICAO_A + LICAO_B))
        dizer("A12")
        dizer("N30")
        if escolher({"W": "entendi", "S": "ouvir de novo"}) == "W":
            break
    dizer("N31")
    aprender_magia("soma")

## Cena 4 — O Grilo Falante
**Vozes:** Alquimista, Narrador, Grilo. O Grilo é o guia: ele **treina os comandos** com o jogador.

In [ ]:
FALAS.update({
    "A13": (ALQUIMISTA, "Agora vá. Saia pela porta, siga a trilha, e a floresta começa logo depois. "
                        "Se ouvir uma vozinha por aí, não se assuste: é o grilo falante. "
                        "Ele guia quem passa por essa trilha."),
    "N40": (NARRADOR, "Na trilha, o silêncio dura pouco. Um cri-cri alegre vem da esquerda, "
                      "e uma vozinha se aproxima:"),
    "G01": (GRILO, "Olá, jovem mago! Eu sou o Grilo Falante, seu guia e narrador nesta jornada. "
                   "Pode contar comigo para saber onde você está e para onde ir... "
                   "e para falar bastante, claro."),
    "N41": (NARRADOR, "É um pequeno grilo que pula de galho em galho, falando mais do que o seu "
                      "tamanho sugere."),
    "G02": (GRILO, "Antes de tudo, os seus comandos. A é esquerda, D é direita, W é frente e S é trás. "
                   "R repete o que eu acabei de dizer, O diz quais teclas valem agora, P pausa, V conta suas vidas "
                   "e H lembra os comandos. "
                   "Se eu estiver falando rápido ou devagar demais, use as teclas mais e menos. "
                   "Vamos treinar! Aperte A para pular para a esquerda."),
    "G03": (GRILO, "Isso! Agora D, para a direita."),
    "G04": (GRILO, "Muito bem! Agora S, um passo para trás."),
    "G05": (GRILO, "Perfeito! E W para seguir em frente."),
    "G06": (GRILO, "Você aprende rápido! Mais à frente, a trilha se divide. Dois caminhos, jovem mago. "
                   "Um tem pássaros. O outro tem silêncio. Eu escolheria os pássaros."),
})

In [ ]:
def cena_4_grilo():
    dizer("A13")
    som("cri-cri alegre vindo da esquerda", "grilo")
    dizer("N40")
    dizer("G01")
    dizer("N41")
    dizer("G02")
    escolher({"A": "pular para a esquerda"})
    som("passos na folhagem, à esquerda", "passo")
    dizer("G03")
    escolher({"D": "pular para a direita"})
    som("passos na folhagem, à direita", "passo")
    dizer("G04")
    escolher({"S": "dar um passo para trás"})
    som("passos para trás", "passo")
    dizer("G05")
    escolher({"W": "seguir em frente"})
    dizer("G06")

## Cena 5 — A Bifurcação
**Vozes:** Narrador, Voz misteriosa, Grilo. O caminho vermelho é uma armadilha: o jogador volta para a bifurcação (sem perder vida).

In [ ]:
FALAS.update({
    "N50": (NARRADOR, "Você chega ao ponto em que a trilha se divide."),
    "N51": (NARRADOR, "À esquerda, o caminho da floresta é cheio de vida: pássaros, folhas e um "
                      "riacho correndo ao longe."),
    "N52": (NARRADOR, "À direita, o caminho vermelho tem terra cor de ferrugem, vento seco e nenhum bicho."),
    "N53": (NARRADOR, "Aperte A para o caminho da floresta, ou D para o caminho vermelho."),
    "V01": (VOZ_MISTERIOSA, "Volte..."),
    "N54": (NARRADOR, "Do fundo do caminho vermelho, um sussurro. Aperte S para voltar, ou W para insistir."),
    "N55": (NARRADOR, "O chão cede sob seus pés, tudo escurece... e você desperta de novo na bifurcação."),
    "G07": (GRILO, "Eu avisei! Pássaros, jovem mago. Pássaros."),
    "N56": (NARRADOR, "Você volta para a bifurcação."),
    "N57": (NARRADOR, "Você segue pela trilha da floresta. Os pássaros cantam acima de você."),
})

In [ ]:
def cena_5_bifurcacao():
    dizer("N50")
    while True:
        som("pássaros e um riacho à esquerda; vento seco à direita", "bifurcacao")
        dizer("N51")
        dizer("N52")
        dizer("N53")
        if escolher({"A": "o caminho da floresta", "D": "o caminho vermelho"}) == "A":
            break
        som("vento seco, nenhum bicho", "vento_seco")
        dizer("V01")
        dizer("N54")
        if escolher({"S": "voltar", "W": "insistir"}, certa="S") == "W":
            som("terra desmoronando", "desmoronamento")
            dizer("N55")
            dizer("G07")
        else:
            dizer("N56")
    dizer("N57")

## Cena 6 — O Goblin (tutorial de combate)
**Vozes:** Narrador, Goblin, Grilo.
1. **Esquiva pelo som:** o golpe vem da esquerda, da direita ou de frente; o jogador pula para o lado contrário. Errar tira 1 vida.
2. **Pedágio:** somar as matrizes A e B (escolha entre 3 respostas). As erradas são erros comuns (subtração e multiplicação posição a posição), e o Grilo explica o erro.

> No notebook não existe som estéreo nem tempo limite (o `input()` espera). No `.exe` com pygame, o golpe toca no canal esquerdo/direito e tem tempo para reagir.

In [ ]:
FALAS.update({
    "N60": (NARRADOR, "Na floresta, uma risada aguda circula você, rápida demais para ser seguida. "
                      "Um goblin salta à sua frente!"),
    "GB01": (GOBLIN, "Quem pisa na minha floresta paga pedágio! Mas antes... vamos ver se você é rápido!"),
    "G08": (GRILO, "Cuidado! Escute de onde vem o golpe e pule para o lado contrário. "
                   "Golpe da esquerda, aperte D. Golpe da direita, aperte A. Golpe de frente, aperte S para recuar."),
    "N61_esquerda": (NARRADOR, "O golpe vem da esquerda!"),
    "N61_direita": (NARRADOR, "O golpe vem da direita!"),
    "N61_frente": (NARRADOR, "O golpe vem de frente!"),
    "GB02": (GOBLIN, "Grrr! Escorregadio! Então paga com a cabeça!"),
    "GB03": (GOBLIN, "Duas matrizes, A e B. Some as duas e eu deixo você passar! "
                     "A: dois e um na primeira linha, zero e três na segunda. "
                     "B: um e quatro na primeira linha, cinco e dois na segunda. Quero a soma!"),
    "G09": (GRILO, "Lembra da lição: posição com posição. Escute as três respostas."),
    "GB06": (GOBLIN, "Qual é a soma de A mais B? Hein? Hein?"),
    "GB04": (GOBLIN, "Três, cinco, cinco, cinco! Grrr! Passa, passa, mago chato!"),
    "N62": (NARRADOR, "E some correndo entre as árvores."),
    "GB05": (GOBLIN, "Errado! Hahaha! Paga de novo!"),
    "G10_subtracao": (GRILO, "Ops! Essa foi a subtração, não a soma. Na soma, comece pelo canto "
                             "de cima, à esquerda: dois mais um."),
    "G10_multiplicacao": (GRILO, "Ops! Aí você multiplicou posição com posição, não somou. Na soma, "
                                 "comece pelo canto de cima, à esquerda: dois mais um."),
    "N63": (NARRADOR, "Tudo escurece... Você respira fundo e tenta de novo."),
    "G11": (GRILO, "De volta à floresta. Vamos lá, jovem mago, dessa vez vai!"),
})

In [ ]:
GOBLIN_A = np.array([[2, 1], [0, 3]])
GOBLIN_B = np.array([[1, 4], [5, 2]])

# lado do golpe -> tecla certa para esquivar (pular para o lado contrário)
LADOS_GOLPE = {"esquerda": "D", "direita": "A", "frente": "S"}
ESQUIVAS = {"A": "pular para a esquerda", "W": "avançar", "S": "recuar", "D": "pular para a direita"}


def esquiva(som_ataque="passos rápidos e um grito", arquivo="golpe"):
    """Um ataque vem de um lado; o jogador pula para o lado contrário.
    Usada pelo Goblin e pelo Dragão (cada um com o seu som)."""
    lado = random.choice(list(LADOS_GOLPE))
    tecla_certa = LADOS_GOLPE[lado]
    som(f"{som_ataque} vindo da {lado}", f"{arquivo}_{lado}")
    dizer(f"N61_{lado}")
    if escolher(ESQUIVAS, certa=tecla_certa) == tecla_certa:
        som("o golpe passa raspando", "esquiva")
        dizer("S10")
    else:
        perder_vida()


def pedagio_goblin():
    opcoes = [                       # resposta certa + dois erros comuns
        ("soma", GOBLIN_A + GOBLIN_B),
        ("subtracao", GOBLIN_A - GOBLIN_B),
        ("multiplicacao", GOBLIN_A * GOBLIN_B),
    ]
    dizer("GB03")
    print("    A =\n" + desenhar_matriz(GOBLIN_A))
    print("    B =\n" + desenhar_matriz(GOBLIN_B))
    dizer("G09")
    while True:
        random.shuffle(opcoes)
        certo = next(i for i, (nome, _) in enumerate(opcoes) if nome == "soma")
        escolha = perguntar_matrizes("GB06", [M for _, M in opcoes], certo)
        if escolha == certo:
            dizer("S07")
            return
        dizer("GB05")
        dizer(f"G10_{opcoes[escolha][0]}")
        perder_vida()


def cena_6_goblin():
    estado.checkpoint = cena_6_goblin
    if estado.retorno:
        estado.retorno = False
        dizer("N63")
        dizer("G11")
    som("risada aguda circulando ao seu redor", "goblin_risada")
    dizer("N60")
    dizer("GB01")
    dizer("G08")
    for _ in range(3):
        esquiva()
    dizer("GB02")
    pedagio_goblin()
    dizer("GB04")
    som("passos correndo entre as árvores", "goblin_foge")
    dizer("N62")

## Cena 7 — A Praia e o Checkpoint do Alquimista
**Vozes:** Narrador, Alquimista. **Matemática:** matriz triangular superior. A partir daqui, perder todas as vidas leva de volta à praia.

In [ ]:
FALAS.update({
    "N70": (NARRADOR, "Ondas, gaivotas e areia sob os pés. A voz do Alquimista chega por telepatia, "
                      "com um leve eco:"),
    "A20": (ALQUIMISTA, "Chegou à praia? Ótimo. Aqui você aprende a Matriz Triangular superior: "
                        "tem zero em todas as posições abaixo da diagonal principal."),
    "A21": (ALQUIMISTA, "A diagonal principal vai do canto de cima, à esquerda, até o canto de baixo, "
                        "à direita. Por exemplo: um e dois em cima, zero e três embaixo. O zero fica no "
                        "canto de baixo, à esquerda. Ela corta caminho no meio da tempestade, e você "
                        "vai precisar dela."),
    "N71": (NARRADOR, "Aperte W se entendeu, ou S para ouvir a explicação de novo."),
    "N72": (NARRADOR, "A pedra pulsa de novo."),
    "A22": (ALQUIMISTA, "Este é o meu checkpoint. Se der algo errado, você volta para cá."),
})

In [ ]:
EXEMPLO_TRIANGULAR = np.array([[1, 2], [0, 3]])


def cena_7_praia():
    som("ondas e gaivotas", "praia")
    dizer("N70")
    dizer("A20")
    while True:
        dizer("A21")
        print(desenhar_matriz(EXEMPLO_TRIANGULAR))
        dizer("N71")
        if escolher({"W": "entendi", "S": "ouvir de novo"}) == "W":
            break
    dizer("N72")
    aprender_magia("triangular")
    dizer("A22")
    estado.checkpoint = cena_8_barco

## Cena 8 — O Barco e a Tempestade
**Vozes:** Narrador, Alquimista. **Desafio:** 3 relâmpagos; em cada um, 3 matrizes aleatórias e só uma é triangular superior.
Errar tira 1 vida e sorteia novas matrizes. Perdeu as 3 vidas? Volta à praia ("De novo.").
Os três relâmpagos usam matrizes 2x2 (quatro números), para a pergunta ser curta de ouvir.

In [ ]:
FALAS.update({
    "N80": (NARRADOR, "Tudo escurece... e você acorda na praia. Na sua cabeça, a voz do Alquimista suspira:"),
    "A23": (ALQUIMISTA, "De novo."),
    "N81": (NARRADOR, "Um barco de madeira, esquecido na areia. Aperte W para empurrá-lo para a água."),
    "N82": (NARRADOR, "Você embarca sozinho. O mar está calmo, só o rangido dos remos. Até que o vento "
                      "muda, o céu escurece e o primeiro trovão estoura."),
    "A24": (ALQUIMISTA, "Tempestade! Foi para isso que te dei a Triangular superior! A cada relâmpago, três "
                        "matrizes vão aparecer. Ache a que tem zero embaixo da diagonal!"),
    "N83_1": (NARRADOR, "Primeiro relâmpago!"),
    "N83_2": (NARRADOR, "Segundo relâmpago!"),
    "N83_3": (NARRADOR, "Terceiro relâmpago! O mais forte de todos!"),
    "N87": (NARRADOR, "Qual destas é triangular superior?"),
    "N84": (NARRADOR, "O barco corta a onda e segue em frente."),
    "N85": (NARRADOR, "Uma onda gigante acerta o barco!"),
    "N86": (NARRADOR, "O vento se acalma, e a proa bate em areia firme: a ilha."),
})

In [ ]:
RELAMPAGOS = [2, 2, 2]   # tamanho das matrizes em cada relâmpago (2x2: 4 números, rápido de ouvir)


def gerar_relampago(n):
    """3 opções: triangular superior (certa), triangular inferior e matriz cheia."""
    opcoes = [np.triu(matriz_aleatoria(n)), np.tril(matriz_aleatoria(n)), matriz_aleatoria(n)]
    random.shuffle(opcoes)
    certo = next(i for i, M in enumerate(opcoes) if e_triangular_superior(M))
    return opcoes, certo


def relampago(numero_relampago, n):
    while True:
        som("um trovão estoura", "trovao")
        dizer(f"N83_{numero_relampago}")
        opcoes, certo = gerar_relampago(n)
        escolha = perguntar_matrizes("N87", opcoes, certo)
        justificativa = justificar_triangular(opcoes[escolha])
        anunciar(NARRADOR, justificativa.texto, justificativa.pecas)
        if escolha == certo:
            dizer("S07")
            dizer("N84")
            return
        dizer("N85")
        perder_vida()


def cena_8_barco():
    if estado.retorno:
        estado.retorno = False
        dizer("N80")
        dizer("A23")
    dizer("N81")
    escolher({"W": "empurrar o barco"}, aceita_enter=True)
    som("madeira raspando na areia; remos rangendo", "barco")
    dizer("N82")
    som("vento forte e ondas batendo no casco", "tempestade")
    dizer("A24")
    for i, n in enumerate(RELAMPAGOS, start=1):
        relampago(i, n)
    som("o vento se acalma", "calmaria")
    dizer("N86")

## Cena 9 — A Ilha e o Farol
**Vozes:** Narrador, Alquimista, Grilo.
- Na **DEMO**, o jogo termina aqui, diante da porta da masmorra (`fim_da_demo`).
- No **jogo completo**, o Alquimista deixa um **checkpoint no farol**: perder todas as vidas na masmorra traz o jogador de volta para cá.
- Também no jogo completo, o Alquimista ensina a **Matriz Diagonal**, a **Matriz Identidade** e a regra do **tipo mais específico** (uma matriz pode ser de vários tipos; a identidade também é diagonal).

In [ ]:
FALAS.update({
    "N90": (NARRADOR, "Silêncio, só o vento e o barulho distante do mar. O farol gira devagar, "
                      "com um som de engrenagem velha."),
    "A25": (ALQUIMISTA, "O dragão está lá no fundo. O farol é antigo e cheio de guardiões. "
                        "Cuidado onde pisa."),
    "N91": (NARRADOR, "Diante de você está a porta pesada da masmorra, sob o farol. "
                      "Lá dentro, gotas ecoam em corredores longos."),
    "G12": (GRILO, "Daqui para frente, jovem mago, nem eu sei o que vem. Mas eu vou junto!"),
    "A26": (ALQUIMISTA, "Vou deixar um checkpoint aqui no farol. Se a masmorra te derrubar, "
                        "você volta para esta porta. E lembre: o dragão teme o zero."),
    "A31": (ALQUIMISTA, "Antes de entrar, mais duas magias. A Matriz Diagonal só tem números na diagonal "
                        "principal; fora dela, tudo é zero. Por exemplo: dois e zero em cima, zero e cinco embaixo. "
                        "Num dois por dois, escute os dois números de fora: o de cima à direita e o de baixo "
                        "à esquerda. Se os dois forem zero, é diagonal."),
    "A32": (ALQUIMISTA, "E a Matriz Identidade é a diagonal mais simples de todas: um em toda a diagonal e zero "
                        "no resto. Um e zero em cima, zero e um embaixo. Ela é o número um das matrizes."),
    "A33": (ALQUIMISTA, "Agora, preste atenção: uma matriz pode ser de vários tipos ao mesmo tempo. A identidade "
                        "também é diagonal, e a nula é tudo isso junto. Quando alguém lá dentro perguntar o tipo, "
                        "diga sempre o mais específico."),
    "N92": (NARRADOR, "Fim da DEMO de Mathrix Souls. A masmorra, com seus guardiões e o dragão, "
                      "espera por você no jogo completo."),
    "N93": (NARRADOR, "Obrigado por jogar! Aperte W para voltar ao menu."),
})

In [ ]:
def cena_9_farol():
    som("vento, mar distante e uma engrenagem velha girando", "farol")
    dizer("N90")
    dizer("A25")
    dizer("N91")
    dizer("G12")
    if estado.modo == "completo":
        dizer("A26")
        licao_diagonal_identidade()
    continuar()


EXEMPLO_DIAGONAL = np.array([[2, 0], [0, 5]])
EXEMPLO_IDENTIDADE = np.eye(2, dtype=int)


def licao_diagonal_identidade():
    """Só no jogo completo: o Alquimista ensina diagonal, identidade e a regra do 'mais específico'."""
    while True:
        dizer("A31")
        print(desenhar_matriz(EXEMPLO_DIAGONAL))
        dizer("A32")
        print(desenhar_matriz(EXEMPLO_IDENTIDADE))
        dizer("A33")
        dizer("N71")
        if escolher({"W": "entendi", "S": "ouvir de novo"}) == "W":
            break
    aprender_magia("diagonal")
    aprender_magia("identidade")


def fim_da_demo():
    dizer("N92")
    dizer(f"S06_{estado.vidas}")
    dizer("N93")
    escolher({"W": "voltar ao menu"}, aceita_enter=True)

---
# Parte 3B — Jogo completo: a masmorra
A partir daqui, só no **jogo completo**. Todas as cenas seguem o roteiro (`Material/roteirorascunho.pdf`).
A magia nova é a **Matriz Nula** (todos os números são zero). Os dois caminhos da masmorra a ensinam, e é ela que derrota o Dragão.

## Cena 10 — A Masmorra
**Vozes:** Narrador, Voz misteriosa (as paredes), Grilo, Alquimista. O jogador escolhe o corredor: **A** = armadura (Cavaleiro Fantasma) ou **D** = pedra (Golem). Os dois levam ao Dragão.

In [ ]:
FALAS.update({
    "N100": (NARRADOR, "Você empurra a porta pesada e entra na masmorra. Gotas ecoam em corredores longos."),
    "V02": (VOZ_MISTERIOSA, "Há muito tempo, magos de matrizes construíram este farol para guiar os navios no escuro. "
                            "Então veio o dragão, e o cheiro dele corrompeu os guardiões. Agora eles testam todo "
                            "aquele que se aproxima."),
    "N101": (NARRADOR, "O corredor se divide. À esquerda, o rangido de uma armadura. À direita, passos "
                       "pesados de pedra. Os dois caminhos levam ao dragão."),
    "G13": (GRILO, "Dessa vez nem eu sei qual é melhor. Os dois têm cara de encrenca."),
    "N102": (NARRADOR, "Aperte A para o corredor da armadura, ou D para o corredor de pedra."),
    "N103": (NARRADOR, "Tudo escurece... e você desperta diante da porta do farol, ao som da voz do Alquimista:"),
    "A27": (ALQUIMISTA, "De novo. Ainda bem que deixei o checkpoint aqui. Respire e entre outra vez."),
})

In [ ]:
def cena_10_masmorra():
    estado.checkpoint = cena_10_masmorra
    if estado.retorno:
        estado.retorno = False
        dizer("N103")
        dizer("A27")
    som("porta pesada rangendo; gotas ecoando", "masmorra")
    dizer("N100")
    dizer("V02")
    som("à esquerda, o rangido de uma armadura; à direita, passos de pedra", "corredores")
    dizer("N101")
    dizer("G13")
    dizer("N102")
    tecla = escolher({"A": "o corredor da armadura", "D": "o corredor de pedra"})
    estado.caminho = "cavaleiro" if tecla == "A" else "golem"

## Cena 11A — O Cavaleiro Fantasma e o Banquete
**Vozes:** Cavaleiro Fantasma, Narrador.
- O banquete é uma ilusão: **comer tira 1 vida**; **recusar** não custa nada. Nos dois casos o Cavaleiro entrega a **Matriz Nula** (como no roteiro).
- **Teste do zero:** somar a Matriz Nula a uma matriz A. O resultado é a própria A, porque a nula é o **elemento neutro da soma** (A + 0 = A).
  Respostas erradas são erros comuns: achar que "tudo vira zero", ou que somar zero aumenta os números.- **O espelho:** o reflexo do Cavaleiro é a **transposta** (linhas viram colunas). O jogador escolhe a transposta de uma matriz sorteada; os erros são a própria matriz e a matriz "de cabeça para baixo".
  Depois, só atravessa o espelho quem é igual ao próprio reflexo: a **simétrica** (A = Aᵀ). As outras opções são uma triangular e uma matriz qualquer, sem simetria.

In [ ]:
FALAS.update({
    "CF01": (CAVALEIRO, "Sou o Cavaleiro Fantasma. Há séculos guardo este corredor. Você deve estar "
                        "faminto, jovem mago. Venha, sente-se."),
    "N110": (NARRADOR, "Uma mesa gigante surge do nada, com cheiro de carne assada, pão e frutas doces."),
    "CF02": (CAVALEIRO, "Este é o Banquete Gigante. Coma comigo. Ou recuse, se tiver coragem."),
    "N111": (NARRADOR, "Aperte W para comer, ou S para recusar."),
    "CF03": (CAVALEIRO, "Poucos resistem. Quem recusa merece a Matriz Nula: aquela em que todos os "
                        "números são zero. É a única coisa que o dragão teme."),
    "N112": (NARRADOR, "O primeiro pedaço é doce e gelado, e você sente um pedaço da sua energia escapar."),
    "CF04": (CAVALEIRO, "Ilusão cara, jovem mago! Mesmo assim, tome a Matriz Nula: aquela em que todos "
                        "os números são zero. Você vai precisar dela mais do que eu."),
    "CF05": (CAVALEIRO, "Mas antes, mostre que entende o zero. Some a Matriz Nula a esta matriz: "
                        "três e um em cima, dois e quatro embaixo."),
    "CF06": (CAVALEIRO, "Qual é o resultado da soma?"),
    "CF07": (CAVALEIRO, "Isso. Somar zero não muda nada: a matriz continua a mesma. A Matriz Nula é "
                        "o elemento neutro da soma."),
    "CF08_nula": (CAVALEIRO, "Não. Somar zero não apaga a matriz: três mais zero continua três."),
    "CF08_mais_um": (CAVALEIRO, "Não. Somar zero não acrescenta nada: três mais zero é três, não quatro."),
    "CF09": (CAVALEIRO, "Agora, o espelho. Meu reflexo é a Matriz Transposta: as linhas viram colunas. "
                        "Um e dois em cima, três e quatro embaixo, no espelho vira um e três em cima, "
                        "dois e quatro embaixo."),
    "CF16": (CAVALEIRO, "O segredo é simples: a diagonal, do canto de cima à esquerda até o de baixo à "
                        "direita, fica parada. Só os dois números de fora trocam de lugar: o de cima à direita "
                        "vai para baixo à esquerda, e o de baixo à esquerda vai para cima à direita."),
    "N150": (NARRADOR, "Diante do espelho, a matriz:"),
    "CF10": (CAVALEIRO, "Qual é o reflexo dela no espelho, a transposta?"),
    "CF11_igual": (CAVALEIRO, "Não. Essa é a própria matriz. No reflexo, a primeira linha vira a primeira coluna."),
    "CF11_giro": (CAVALEIRO, "Não. Aí você virou a matriz de cabeça para baixo. No reflexo, a primeira linha "
                             "vira a primeira coluna."),
    "CF12": (CAVALEIRO, "Isso. Só atravessa o espelho quem é igual ao próprio reflexo: a Matriz Simétrica. "
                        "Para saber se é simétrica, escute só os dois números de fora da diagonal: o de cima "
                        "à direita e o de baixo à esquerda. Se forem iguais, ela é simétrica."),
    "CF13": (CAVALEIRO, "Qual destas atravessa o espelho?"),
    "CF14_triangular": (CAVALEIRO, "Não. Ela tem zero de um lado da diagonal e números do outro: o reflexo "
                                   "seria diferente."),
    "CF14_cheia": (CAVALEIRO, "Não. Compare os números que trocam de lugar no espelho: eles são diferentes."),
    "CF15": (CAVALEIRO, "Igual ao próprio reflexo. Pode passar, jovem mago."),
    "N113": (NARRADOR, "O espelho se apaga, e o corredor se abre até a porta do dragão."),
})

In [ ]:
CAVALEIRO_A = np.array([[3, 1], [2, 4]])
NULA_2x2 = np.zeros((2, 2), dtype=int)


def cena_11a_cavaleiro():
    som("o rangido de uma armadura vazia", "armadura")
    dizer("CF01")
    som("pratos, talheres e cheiro de comida", "banquete")
    dizer("N110")
    dizer("CF02")
    dizer("N111")
    if escolher({"W": "comer", "S": "recusar"}, certa="S") == "S":
        dizer("CF03")
    else:
        dizer("N112")
        perder_vida()
        dizer("CF04")
    dizer("CF05")
    print(desenhar_matriz(CAVALEIRO_A), "\n        +")
    print(desenhar_matriz(NULA_2x2))
    desafio_matrizes("CF06", [
        ("certa", CAVALEIRO_A + NULA_2x2),   # A + 0 = A
        ("nula", NULA_2x2),                  # erro: "tudo vira zero"
        ("mais_um", CAVALEIRO_A + 1),        # erro: "somar aumenta"
    ], prefixo_erro="CF08")
    dizer("CF07")
    aprender_magia("nula")
    espelho_do_cavaleiro()
    dizer("N113")


def espelho_do_cavaleiro():
    """Transposta (o reflexo) e simétrica (igual ao próprio reflexo)."""
    som("um espelho antigo vibrando", "espelho")
    dizer("CF09")
    dizer("CF16")
    while True:   # sem simetria e com a diagonal diferente: assim as 3 opções são todas diferentes
        M = matriz_aleatoria(2)
        if not e_simetrica(M) and M[0, 0] != M[1, 1]:
            break
    leitura = ler_matriz(M, Frase().add("N150"))
    anunciar(NARRADOR, leitura.texto, leitura.pecas)
    print(desenhar_matriz(M))
    desafio_matrizes("CF10", [
        ("certa", M.T),            # transposta: linhas viram colunas
        ("igual", M),              # erro: não refletiu
        ("giro", M[::-1, ::-1]),   # erro: virou de cabeça para baixo
    ], prefixo_erro="CF11")
    aprender_magia("transposta")
    dizer("CF12")
    desafio_matrizes("CF13", [
        ("certa", gerar_matriz_do_tipo("simetrica")),
        ("triangular", gerar_matriz_do_tipo("triangular")),
        ("cheia", gerar_matriz_do_tipo("cheia")),
    ], prefixo_erro="CF14")
    dizer("CF15")
    aprender_magia("simetrica")

## Cena 11B — O Golem de Pedra e o Puzzle das Pedras
**Vozes:** Golem, Grilo, Narrador.
1. **Pergunta do Golem** (do roteiro): a matriz quatro e sete em cima, dois e nove embaixo, **menos ela mesma**. A resposta é a **Matriz Nula** (A − A = 0).
   Respostas erradas: a própria matriz (achar que subtrair dela mesma não muda) e o dobro (somar em vez de subtrair).
2. **Puzzle das pedras** (do MVP, estilo "Squid Game"): 3 saltos sobre um abismo. Em cada um, o Golem diz qual matriz aguenta o peso: **nula**, depois **identidade**, depois **diagonal** (todas 2x2). Das três pedras (A, W, D), só uma é do tipo pedido; as outras são de tipos parecidos que **não** servem (ex.: no salto da identidade, uma diagonal qualquer e a nula). Errar tira 1 vida, e a justificativa diz que tipo era a pedra.

In [ ]:
FALAS.update({
    "N115": (NARRADOR, "O chão treme. Pedras se juntam, rangendo, até formar uma voz:"),
    "GO01": (GOLEM, "Eu sou o Golem. Eu sou pedra. Eu guardo este caminho."),
    "GO02": (GOLEM, "Mostro uma matriz: quatro e sete em cima, dois e nove embaixo. Diga o que "
                    "sobra quando ela é subtraída dela mesma."),
    "G14": (GRILO, "Posição com posição, jovem mago. Quatro menos quatro..."),
    "GO03": (GOLEM, "O que sobra?"),
    "GO04": (GOLEM, "Zero. Tudo zero. Você entendeu a Matriz Nula."),
    "G15_igual": (GRILO, "Ops! Essa é a mesma matriz. Subtrair uma matriz dela mesma não deixa nada sobrando."),
    "G15_dobro": (GRILO, "Ops! Aí você somou a matriz com ela mesma. Subtrair é tirar: quatro menos quatro dá zero."),
    "GO05": (GOLEM, "Mas o caminho ainda não é seu. Atravesse o abismo. Em cada salto, eu digo qual "
                    "matriz aguenta o seu peso. As outras afundam."),
    "G16": (GRILO, "Em cada salto há três pedras: à esquerda, no meio e à direita. Escute o Golem e "
                   "pule na pedra certa!"),
    "GO07_nula": (GOLEM, "Pise na Matriz Nula."),
    "GO07_identidade": (GOLEM, "Pise na Matriz Identidade."),
    "GO07_diagonal": (GOLEM, "Pise na Matriz Diagonal."),
    "G18_nula": (GRILO, "Dica: na nula, os quatro números são zero."),
    "G18_identidade": (GRILO, "Dica: a identidade é um e zero em cima, zero e um embaixo."),
    "G18_diagonal": (GRILO, "Dica: na diagonal, os dois números de fora, o de cima à direita e o de baixo à "
                            "esquerda, são zero."),
    "N116_1": (NARRADOR, "Primeiro salto."),
    "N116_2": (NARRADOR, "Segundo salto."),
    "N116_3": (NARRADOR, "Último salto. O vento do abismo sobe forte."),
    "N117": (NARRADOR, "Em qual pedra ela está?"),
    "N118": (NARRADOR, "Você pula, e a pedra aguenta firme."),
    "N119": (NARRADOR, "A pedra afunda sob seus pés! Você se agarra na borda e volta."),
    "GO06": (GOLEM, "Você é digno. Passe."),
    "N120": (NARRADOR, "O golem se desfaz em poeira, abrindo o caminho até a porta do dragão."),
})

In [ ]:
GOLEM_A = np.array([[4, 7], [2, 9]])
# Em cada salto, o Golem pede um tipo; as outras pedras são de tipos que NÃO servem.
# (tipo pedido, tamanho, tipos das pedras erradas)
SALTOS = [
    ("nula", 2, ["identidade", "diagonal"]),
    ("identidade", 2, ["diagonal", "nula"]),
    ("diagonal", 2, ["triangular", "simetrica"]),
]


def gerar_pedras(alvo, n, distratores):
    pedras = [gerar_matriz_do_tipo(alvo, n)] + [gerar_matriz_do_tipo(t, n) for t in distratores]
    random.shuffle(pedras)
    certo = next(i for i, M in enumerate(pedras) if E_DO_TIPO[alvo](M))
    return pedras, certo


def salto(numero_salto, alvo, n, distratores):
    while True:
        som("vento subindo do abismo", "abismo")
        dizer(f"N116_{numero_salto}")
        dizer(f"GO07_{alvo}")
        dizer(f"G18_{alvo}")
        pedras, certo = gerar_pedras(alvo, n, distratores)
        escolha = perguntar_matrizes("N117", pedras, certo)
        justificativa = justificar_tipo(pedras[escolha], alvo)
        anunciar(NARRADOR, justificativa.texto, justificativa.pecas)
        if escolha == certo:
            som("pés firmes na pedra", "pedra_firme")
            dizer("N118")
            return
        som("pedra afundando", "pedra_afunda")
        dizer("N119")
        perder_vida()


def cena_11b_golem():
    som("o chão treme; pedras rangendo", "golem")
    dizer("N115")
    dizer("GO01")
    dizer("GO02")
    print(desenhar_matriz(GOLEM_A), "\n        -")
    print(desenhar_matriz(GOLEM_A))
    dizer("G14")
    desafio_matrizes("GO03", [
        ("certa", GOLEM_A - GOLEM_A),   # A - A = 0
        ("igual", GOLEM_A),             # erro: "subtrair dela mesma não muda"
        ("dobro", GOLEM_A + GOLEM_A),   # erro: somou em vez de subtrair
    ], prefixo_erro="G15")
    dizer("GO04")
    aprender_magia("nula")
    dizer("GO05")
    dizer("G16")
    for i, (alvo, n, distratores) in enumerate(SALTOS, start=1):
        salto(i, alvo, n, distratores)
    dizer("GO06")
    som("pedras virando poeira", "golem_poeira")
    dizer("N120")


def cena_11_guardiao():
    """Vai para o guardião do corredor escolhido na cena 10."""
    if estado.caminho == "cavaleiro":
        cena_11a_cavaleiro()
    else:
        cena_11b_golem()

## Cena 12 — O Dragão (chefe final)
**Vozes:** Narrador, Dragão, Alquimista.
O fogo do Dragão é feito de matrizes. São 3 rodadas, e cada uma começa com as **garras** (a mesma esquiva do Goblin):
1. **Rodadas 1 e 2 — classificar o fogo** (2x2): o Dragão cospe uma matriz de um tipo que o jogador aprendeu (triangular superior, diagonal, identidade e, se veio pelo Cavaleiro, simétrica), e o jogador diz o **tipo mais específico** entre 3 opções. É o combate de classificação do MVP. Errou: a justificativa explica o tipo certo, inclusive quando a matriz "até é" do tipo escolhido, mas existe um mais específico (ex.: escolheu diagonal para a identidade).
2. **Rodada 3 — o golpe final do roteiro:** para o último fogo M, o jogador conjura **M − M**, a **Matriz Nula**. Erros: a própria M e 2M.

Antes da luta, o Alquimista dá **a receita** (dá para ouvir de novo): num 2x2, basta escutar os dois números de fora da diagonal.
- **Os dois são zero:** é diagonal. Se a diagonal for 1 e 1, é identidade; se tudo for zero, é nula.
- **Só o de baixo é zero:** é triangular superior.
- **Os dois são iguais:** é simétrica.
- **No golpe final:** cada número menos ele mesmo dá zero.

Perder todas as vidas leva de volta ao checkpoint do farol.

In [ ]:
FALAS.update({
    "N121": (NARRADOR, "Os dois caminhos se encontram em uma porta enorme. Do outro lado, uma "
                       "respiração profunda faz o chão tremer, e o ar esquenta."),
    "DR01": (DRAGAO, "Mais um mago... Eles sempre acham que são os primeiros."),
    "A28": (ALQUIMISTA, "É ele! O fogo dele é feito de matrizes. A cada fogo, diga que tipo de matriz ele é, "
                        "sempre o mais específico, e o fogo perde a força. Quando o dragão estiver fraco, "
                        "apague o último fogo com o zero: subtraia a matriz dela mesma! E quando ele atacar "
                        "com as garras, escute de onde vem e pule para o outro lado."),
    "N122_1": (NARRADOR, "O dragão respira fundo e cospe a primeira matriz em chamas!"),
    "N122_2": (NARRADOR, "Furioso, o dragão cospe outra matriz em chamas!"),
    "N122_3": (NARRADOR, "O dragão abre as asas e cospe a maior matriz de todas!"),
    "N123": (NARRADOR, "A matriz em chamas:"),
    "N124": (NARRADOR, "Qual é a matriz dele menos ela mesma?"),
    "N128": (NARRADOR, "Que tipo de matriz é esse fogo?"),
    "N129": (NARRADOR, "Acertou o tipo! O fogo perde a força, e o dragão recua."),
    "A34": (ALQUIMISTA, "Não! Escute o que ele é:"),
    "A35": (ALQUIMISTA, "Ele está fraco! Agora, o último fogo: apague com a Matriz Nula!"),
    "A36": (ALQUIMISTA, "A receita para descobrir o tipo é escutar os dois números de fora da diagonal: "
                        "o de cima à direita e o de baixo à esquerda."),
    "A37": (ALQUIMISTA, "Se os dois forem zero, é diagonal. Mas se a diagonal for um e um, é identidade; "
                        "e se tudo for zero, é nula. Se só o de baixo for zero, é triangular superior. "
                        "E se os dois forem iguais, é simétrica."),
    "A38": (ALQUIMISTA, "No último fogo, a conta é a matriz menos ela mesma: cada número menos ele mesmo dá "
                        "zero. Então a resposta é sempre a matriz em que tudo é zero."),
    "N125": (NARRADOR, "A Matriz Nula apaga o fogo! O dragão ruge de dor."),
    "N126": (NARRADOR, "As chamas te atingem!"),
    "A29_igual": (ALQUIMISTA, "Não! Essa é a própria matriz dele, você não subtraiu nada!"),
    "A29_dobro": (ALQUIMISTA, "Não! Isso é a matriz dele somada com ela mesma: o fogo dobrou!"),
    "N127": (NARRADOR, "O tom do ar desce até quase o silêncio. O dragão ruge, vacila e cai, e a "
                       "masmorra inteira para de tremer."),
    "DR02": (DRAGAO, "Impossível... um mago... que entende o zero..."),
    "N134": (NARRADOR, "Atrás do dragão caído, um monte de moedas de ouro brilha no escuro. É o bastante "
                       "para as ervas da sua mãe, e ainda sobra."),
})

In [ ]:
# Cada rodada: (o que o jogador faz, tamanho da matriz). As duas primeiras revisam os tipos que
# o jogador aprendeu; a última é o golpe final do roteiro: M - M = Matriz Nula.
RODADAS_DRAGAO = [("classificar", 2), ("classificar", 2), ("anular", 2)]   # tudo 2x2
TIPOS_DO_DRAGAO = ["triangular", "diagonal", "identidade", "simetrica"]   # nula fica para o golpe final


def tipos_conhecidos():
    """Os tipos que o jogador já aprendeu (o caminho do Golem não ensina a simétrica)."""
    conhecidos = [t for t in ORDEM_DOS_TIPOS if t in estado.magias]
    return conhecidos if len(conhecidos) >= 3 else ["nula", "identidade", "diagonal", "triangular"]


def rodada_classificar(numero_rodada, n, ja_cobrados):
    """O dragão cospe uma matriz de um tipo que o jogador conhece; ele diz o tipo mais específico."""
    conhecidos = tipos_conhecidos()
    candidatos = [t for t in TIPOS_DO_DRAGAO if t in conhecidos and t not in ja_cobrados] or \
                 [t for t in TIPOS_DO_DRAGAO if t in conhecidos]
    alvo = random.choice(candidatos)
    ja_cobrados.append(alvo)
    esquiva("asas batendo e garras", "garra")
    while True:
        som("um rugido e o estalo de chamas", "fogo")
        dizer(f"N122_{numero_rodada}")
        M = gerar_matriz_do_tipo(alvo, n)
        leitura = ler_matriz(M, Frase().add("N123"))
        anunciar(NARRADOR, leitura.texto, leitura.pecas)
        print(desenhar_matriz(M))
        tipos = opcoes_de_tipo(alvo, conhecidos)
        frases = [Frase().add(f"OPCAO_{t}").add(f"TIPO_{tipo}") for t, tipo in zip(TECLAS_OPCOES, tipos)]
        escolha = perguntar_opcoes("N128", frases, tipos.index(alvo))
        if tipos[escolha] == alvo:
            justificativa = justificar_tipo(M)
            anunciar(NARRADOR, justificativa.texto, justificativa.pecas)
            som("fogo enfraquecendo", "fogo_apaga")
            dizer("N129")
            return
        dizer("A34")
        justificativa = justificar_tipo(M, tipos[escolha])
        anunciar(NARRADOR, justificativa.texto, justificativa.pecas)
        som("chamas", "queimadura")
        dizer("N126")
        perder_vida()


def rodada_dragao(numero_rodada, n):
    """Golpe final: M - M = Matriz Nula apaga o fogo."""
    esquiva("asas batendo e garras", "garra")
    dizer("A35")
    while True:
        som("um rugido e o estalo de chamas", "fogo")
        dizer(f"N122_{numero_rodada}")
        M = matriz_aleatoria(n)
        leitura = ler_matriz(M, Frase().add("N123"))
        anunciar(NARRADOR, leitura.texto, leitura.pecas)
        print(desenhar_matriz(M))
        opcoes = [
            ("certa", M - M),    # M - M = 0: o zero apaga o fogo
            ("igual", M),        # erro: não subtraiu nada
            ("dobro", M + M),    # erro: somou em vez de subtrair
        ]
        random.shuffle(opcoes)
        certo = next(i for i, (nome, _) in enumerate(opcoes) if nome == "certa")
        escolha = perguntar_matrizes("N124", [X for _, X in opcoes], certo)
        if escolha == certo:
            som("fogo se apagando com um chiado", "fogo_apaga")
            dizer("N125")
            return
        dizer(f"A29_{opcoes[escolha][0]}")
        som("chamas", "queimadura")
        dizer("N126")
        perder_vida()


def cena_12_dragao():
    som("uma respiração profunda e o chão tremendo", "dragao_respira")
    dizer("N121")
    dizer("DR01")
    dizer("A28")
    while True:   # a receita de como resolver, com opção de ouvir de novo
        dizer("A36")
        dizer("A37")
        dizer("A38")
        dizer("N71")
        if escolher({"W": "entendi", "S": "ouvir de novo"}) == "W":
            break
    ja_cobrados = []
    for i, (acao, n) in enumerate(RODADAS_DRAGAO, start=1):
        if acao == "classificar":
            rodada_classificar(i, n, ja_cobrados)
        else:
            rodada_dragao(i, n)
    som("um rugido longo e um corpo enorme caindo", "dragao_cai")
    dizer("N127")
    dizer("DR02")
    som("moedas de ouro tilintando", "tesouro")
    dizer("N134")

## Epílogo — De volta à taverna
**Vozes:** Narrador, Alquimista, Max, Grilo. (O Druida e a Mãe aparecem só na narração, sem voz própria.)
O Alquimista confessa que já tinha mandado outros magos (fechando o "eles sempre acham que são os primeiros" do Dragão). Depois, o herói leva o ouro do Dragão à casa de cura: o Druida compra as ervas, a mãe melhora e o Alquimista finalmente pode sair do vilarejo, tudo contado pelo Narrador.

In [ ]:
FALAS.update({
    "N130": (NARRADOR, "Sem o dragão, o cheiro estranho vai embora, e as criaturas da floresta se acalmam. De "
                       "volta à taverna, o Alquimista finge que nunca duvidou de você:"),
    "A30": (ALQUIMISTA, "Eu sabia que você conseguiria. Quer dizer... quase tinha certeza."),
    "A39": (ALQUIMISTA, "Tá bom, tá bom. Talvez você não tenha sido o primeiro mago que eu mandei para lá. "
                        "Mas você foi o primeiro que entendeu o zero."),
    "N133": (NARRADOR, "Com o ouro do dragão, você corre até a casa de cura. O Druida conta as moedas, sorri e "
                       "vai buscar as ervas raras. Com elas, sua mãe vai ficar boa."),
    "N135": (NARRADOR, "Na cama, ela abre os olhos e sorri. O filho dela virou um mago de verdade."),
    "N136": (NARRADOR, "Com Trim segura de novo, o Druida libera o Alquimista para sair do vilarejo."),
    "N131": (NARRADOR, "No balcão, Max acorda devagar:"),
    "M02": (MAX, "Quem... onde... o que eu perdi?"),
    "G17": (GRILO, "Só a maior aventura de matrizes da história, Max!"),
    "N132": (NARRADOR, "Fim de Mathrix Souls."),
})

In [ ]:
def epilogo():
    som("a taverna: murmúrios, canecas e lareira", "taverna")
    dizer("N130")
    dizer("A30")
    dizer("A39")
    dizer("N131")
    dizer("M02")
    dizer("G17")
    som("a porta da casa de cura se abrindo", "casa_de_cura")
    dizer("N133")
    dizer("N135")
    dizer("N136")
    dizer("N132")
    dizer(f"S06_{estado.vidas}")
    dizer("N93")
    escolher({"W": "voltar ao menu"}, aceita_enter=True)

---
# Parte 4 — Menu e início do jogo

**Menu principal:** **A** = DEMO · **D** = jogo completo · **C** = seleção de cenas · **S** = como jogar · **W** = velocidade da fala · **G** = vozes gravadas ou do computador · **Q** = sair.

**Vozes (G):** alterna entre **vozes gravadas** (o padrão: toca a gravação de cada fala e, se faltar alguma, a voz do computador completa só aquela) e **só as vozes do computador** (ignora as gravações). Os efeitos sonoros tocam nos dois modos.

**Seleção de cenas (C):** **S** = próxima cena, **W** = anterior, **D** = jogar a partir dela, **A** = voltar ao menu. O jogo segue da cena escolhida até o fim do jogo completo, com 3 vidas. A cena 11 aparece como 11A (Cavaleiro) e 11B (Golem). Se o jogador perder todas as vidas antes de passar por um checkpoint, volta para a cena escolhida. Útil para testar uma cena específica e para quem quer rejogar uma parte.

In [ ]:
CENAS_INICIO = [
    cena_1_casa,
    cena_2_taverna,
    cena_3_licao,
    cena_4_grilo,
    cena_5_bifurcacao,
    cena_6_goblin,
    cena_7_praia,
    cena_8_barco,
    cena_9_farol,
]
ROTEIRO_DEMO = CENAS_INICIO + [fim_da_demo]
ROTEIRO_COMPLETO = CENAS_INICIO + [cena_10_masmorra, cena_11_guardiao, cena_12_dragao, epilogo]


import json as _json

ARQUIVO_DE_SAVE = None   # None = pasta do usuário (%APPDATA%\MathrixSouls); os testes apontam para outro arquivo


def caminho_do_save():
    """Onde fica o jogo salvo. Fica fora da pasta do jogo, para valer também no .exe de entrega
    (que roda de uma pasta temporária). Nos testes automáticos não salva nada, a menos que
    ARQUIVO_DE_SAVE seja definido."""
    if ARQUIVO_DE_SAVE:
        return ARQUIVO_DE_SAVE
    if MODO_TESTE:
        return None
    base = os.environ.get("APPDATA") or os.path.expanduser("~")
    return os.path.join(base, "MathrixSouls", "salvamento.json")


def _cena_por_nome(roteiro, nome):
    return next((c for c in roteiro if c.__name__ == nome), None)


def salvar_jogo(roteiro, i):
    """Guarda onde o jogador está: a cena atual recomeça do início quando ele continuar."""
    arquivo = caminho_do_save()
    if not arquivo:
        return False
    dados = {
        "versao": 1,
        "modo": estado.modo,
        "cena": roteiro[i].__name__,
        "caminho": estado.caminho,
        "magias": estado.magias,
        "vidas": estado.vidas,
        "checkpoint": estado.checkpoint.__name__ if estado.checkpoint else None,
        "velocidade_da_fala": velocidade_da_fala,
    }
    try:
        os.makedirs(os.path.dirname(arquivo), exist_ok=True)
        with open(arquivo, "w", encoding="utf-8") as f:
            _json.dump(dados, f, ensure_ascii=False, indent=1)
        return True
    except OSError:
        return False


def carregar_save():
    """Retorna os dados do jogo salvo, ou None se não existe ou está estragado."""
    arquivo = caminho_do_save()
    if not arquivo or not os.path.exists(arquivo):
        return None
    try:
        with open(arquivo, encoding="utf-8") as f:
            dados = _json.load(f)
        roteiro = ROTEIRO_COMPLETO if dados["modo"] == "completo" else ROTEIRO_DEMO
        if _cena_por_nome(roteiro, dados["cena"]) is None:
            return None
        return dados
    except (OSError, ValueError, KeyError, TypeError):
        return None


def apagar_save():
    arquivo = caminho_do_save()
    if arquivo and os.path.exists(arquivo):
        try:
            os.remove(arquivo)
        except OSError:
            pass


def salvar_ao_voltar(roteiro, i):
    """Tecla M: salva o jogo na cena atual e avisa. No fim da DEMO/epílogo não há o que salvar."""
    if roteiro[i] is epilogo:
        apagar_save()
    if not caminho_do_save() or roteiro[i] in (fim_da_demo, epilogo):
        return
    dizer("SV06" if salvar_jogo(roteiro, i) else "SV05")


def oferecer_salvar(roteiro, i):
    """O jogador apertou Q no meio do jogo: pergunta se quer salvar (W), sair sem salvar (S) ou voltar (A).
    Retorna True se ele voltou ao jogo. Q de novo = sair sem salvar."""
    if roteiro[i] is epilogo:
        apagar_save()                  # chegou ao fim da história: não há o que continuar
    if not caminho_do_save() or roteiro[i] in (fim_da_demo, epilogo):
        return False
    try:
        with pergunta():
            dizer("SV03")
        tecla = escolher({"W": "salvar e sair", "S": "sair sem salvar", "A": "voltar ao jogo"}, certa="W")
        if tecla == "A":
            dizer("SV07")
            return True                # o jogador desistiu de sair
        if tecla == "W":
            dizer("SV04" if salvar_jogo(roteiro, i) else "SV05")
    except (SairDoJogo, KeyboardInterrupt):
        pass
    return False


def continuar_jogo_salvo(dados):
    """Retoma o jogo salvo na cena onde o jogador parou."""
    global velocidade_da_fala
    if dados.get("velocidade_da_fala") in VELOCIDADES_DA_FALA:
        velocidade_da_fala = dados["velocidade_da_fala"]
    roteiro = ROTEIRO_COMPLETO if dados["modo"] == "completo" else ROTEIRO_DEMO
    inicio = roteiro.index(_cena_por_nome(roteiro, dados["cena"]))
    dizer("SV02")
    jogar_roteiro(roteiro[inicio:], dados["modo"], dados.get("caminho"), dados.get("magias"),
                  _cena_por_nome(roteiro, dados.get("checkpoint")), dados.get("vidas"))


def jogar_roteiro(roteiro, modo, caminho=None, magias=None, checkpoint=None, vidas=None):
    """Roda as cenas em ordem. Perdeu todas as vidas: volta ao último checkpoint.
    Se o checkpoint ficou antes do trecho jogado (começou pela seleção de cenas), volta à 1ª cena dele."""
    global estado, em_jogo
    estado = Estado()
    estado.modo = modo
    estado.caminho = caminho
    estado.magias = list(magias or [])
    estado.checkpoint = checkpoint
    if vidas:
        estado.vidas = vidas
    i = 0
    em_jogo = True
    try:
        while i < len(roteiro):
            try:
                roteiro[i]()
                i += 1
            except Derrota:
                estado.vidas = VIDAS_INICIAIS
                estado.retorno = True
                i = roteiro.index(estado.checkpoint) if estado.checkpoint in roteiro else 0
            except SairDoJogo:
                em_jogo = False
                if not oferecer_salvar(roteiro, i):   # Q: pergunta se quer salvar
                    raise
                em_jogo = True                        # A: voltou; a cena atual recomeça
    except VoltarAoMenu:               # M: salva sozinho e volta ao menu principal
        em_jogo = False
        salvar_ao_voltar(roteiro, i)
        return
    finally:
        em_jogo = False
    if modo == "completo":
        apagar_save()                  # terminou o jogo: não há mais o que continuar


# Seleção de cenas: (fala com o nome, cena do ROTEIRO_COMPLETO onde começa, caminho na masmorra)
SELECAO_DE_CENAS = [
    ("SC_1", cena_1_casa, None),
    ("SC_2", cena_2_taverna, None),
    ("SC_3", cena_3_licao, None),
    ("SC_4", cena_4_grilo, None),
    ("SC_5", cena_5_bifurcacao, None),
    ("SC_6", cena_6_goblin, None),
    ("SC_7", cena_7_praia, None),
    ("SC_8", cena_8_barco, None),
    ("SC_9", cena_9_farol, None),
    ("SC_10", cena_10_masmorra, None),
    ("SC_11A", cena_11_guardiao, "cavaleiro"),
    ("SC_11B", cena_11_guardiao, "golem"),
    ("SC_12", cena_12_dragao, None),
    ("SC_EPI", epilogo, None),
]


# Magias que cada cena ensina: quem começa mais adiante pela seleção de cenas já chega sabendo.
MAGIAS_DA_CENA = {
    cena_3_licao: ["soma"],
    cena_7_praia: ["triangular"],
    cena_9_farol: ["diagonal", "identidade"],
    cena_11_guardiao: ["nula"],
}


def selecionar_cena():
    """Lista de cenas: S = próxima, W = anterior, D = jogar a partir dela, A = voltar ao menu.
    A cena escolhida segue até o fim do jogo completo."""
    dizer("N140")
    atual = 0
    dizer(SELECAO_DE_CENAS[atual][0])
    while True:
        tecla = escolher({"S": "próxima cena", "W": "cena anterior", "D": "jogar esta cena",
                          "A": "voltar ao menu"}, certa="D")
        if tecla == "A":
            return
        if tecla == "D":
            _, cena, caminho = SELECAO_DE_CENAS[atual]
            dizer("N141")
            apagar_save()
            inicio = ROTEIRO_COMPLETO.index(cena)
            magias = [m for c in ROTEIRO_COMPLETO[:inicio] for m in MAGIAS_DA_CENA.get(c, [])]
            jogar_roteiro(ROTEIRO_COMPLETO[inicio:], "completo", caminho, magias)
            return
        novo = atual + (1 if tecla == "S" else -1)
        if novo >= len(SELECAO_DE_CENAS):
            dizer("SC_FIM")
        elif novo < 0:
            dizer("SC_INI")
        else:
            atual = novo
            dizer(SELECAO_DE_CENAS[atual][0])


def trocar_vozes():
    """Tecla G no menu: alterna entre vozes gravadas (com a do computador completando as que faltam)
    e só as vozes do computador. Anuncia o modo novo."""
    global usar_gravacoes
    usar_gravacoes = not usar_gravacoes
    dizer("N160" if usar_gravacoes else "N161")


def menu():
    dizer("N01")
    dizer("N02")
    som("um trovão distante e um sino grave", "abertura")
    while True:
        salvo = carregar_save()        # relido a cada volta: o jogador pode ter acabado de salvar com o M
        opcoes = {"A": "jogar a DEMO", "D": "o jogo completo", "S": "ouvir como jogar",
                  "C": "escolher uma cena",
                  "G": "escolher entre vozes gravadas e vozes do computador"}
        with pergunta():
            dizer("N03")
            if salvo:
                dizer("SV01")
                opcoes["E"] = "continuar de onde parou"
        tecla = escolher(opcoes)
        if tecla == "E":
            continuar_jogo_salvo(salvo)
        elif tecla == "G":
            trocar_vozes()
        elif tecla == "C":
            selecionar_cena()
        elif tecla == "A":
            dizer("N06")
            apagar_save()              # jogo novo: o salvo anterior é descartado
            jogar_roteiro(ROTEIRO_DEMO, "demo")
        elif tecla == "D":
            dizer("N04")
            apagar_save()
            jogar_roteiro(ROTEIRO_COMPLETO, "completo")
        else:
            dizer("N05")


def jogar():
    try:
        menu()
    except (SairDoJogo, KeyboardInterrupt):
        dizer("N07")

## ▶ Jogar
Rode a célula abaixo. Digite a tecla e aperte **Enter**.

In [ ]:
jogar()

---
# Apêndice — Roteiro de gravação
Rode a célula abaixo (não precisa jogar) para ver **todas as falas por personagem**, com o nome do arquivo e se já foi gravada (`OK`) ou ainda falta (`FALTA`).
Cada integrante grava as falas do seu personagem e salva em `audios/<CÓDIGO>.wav`. Os efeitos sonoros vão em `audios/sons/<nome>.wav`.

In [ ]:
from collections import defaultdict

SONS = ['vez_do_jogador', 'dano', 'magia', 'galo_vento', 'porta', 'taverna', 'frascos', 'teleporte', 'cajado_sino', 'grilo', 'passo', 'bifurcacao', 'vento_seco', 'desmoronamento', 'golpe_esquerda', 'golpe_direita', 'golpe_frente', 'garra_esquerda', 'garra_direita', 'garra_frente', 'esquiva', 'goblin_risada', 'goblin_foge', 'praia', 'trovao', 'barco', 'tempestade', 'calmaria', 'farol', 'masmorra', 'corredores', 'armadura', 'banquete', 'espelho', 'abismo', 'pedra_firme', 'pedra_afunda', 'golem', 'golem_poeira', 'fogo', 'fogo_apaga', 'queimadura', 'dragao_respira', 'dragao_cai', 'tesouro', 'casa_de_cura', 'abertura']

por_personagem = defaultdict(list)
for codigo, (quem, texto) in FALAS.items():
    por_personagem[quem].append((codigo, texto))

total = gravadas = 0
for quem, falas in por_personagem.items():
    print(f"\n=== {quem} ({len(falas)} falas) ===")
    for codigo, texto in falas:
        ok = achar_audio(codigo) is not None
        total += 1
        gravadas += ok
        print(f"  [{'OK   ' if ok else 'FALTA'}] {codigo}.wav  {texto}")

print(f"\n=== Efeitos sonoros (pasta {PASTA_AUDIOS}/sons) ===")
for nome in SONS:
    ok = achar_audio(nome, os.path.join(PASTA_AUDIOS, "sons")) is not None
    print(f"  [{'OK   ' if ok else 'FALTA'}] {nome}.wav")

print(f"\nFalas gravadas: {gravadas} de {total}.")

---
# Apêndice 2: Gerar o .exe de entrega
Existem dois executáveis:

| | Onde fica | Como funciona |
|---|---|---|
| **Desenvolvimento** | `MathrixSouls.exe`, na pasta principal | Lê **este notebook** e a pasta `audios` toda vez que abre. Mudou uma fala ou o código? **Salve o notebook (Ctrl+S)** e abra o `.exe` de novo. Não precisa gerar outro. |
| **Entrega** | `entrega/MathrixSouls.exe` | Leva o notebook e todas as gravações **dentro dele**. Funciona sozinho, sem nenhuma outra pasta. É o que vai para a pessoa que vai jogar. |

**Para gerar o de entrega:**
1. Salve o notebook (**Ctrl+S**): o `.exe` leva a versão salva.
2. Na célula abaixo, troque `GERAR_EXE_DE_ENTREGA = False` para `True` e rode só ela. Leva de 1 a 2 minutos.
3. Volte para `False` depois (assim o *Run All* não gera o `.exe` sem querer).

Precisa de: `pip install pyinstaller numpy comtypes pygame-ce`.

In [ ]:
GERAR_EXE_DE_ENTREGA = False   # troque para True e rode esta célula

import importlib.util
import subprocess
import sys

if not GERAR_EXE_DE_ENTREGA:
    print("Desligado. Troque GERAR_EXE_DE_ENTREGA para True e rode esta célula de novo.")
elif importlib.util.find_spec("PyInstaller") is None:
    print("Falta o PyInstaller. Rode no terminal: pip install pyinstaller")
else:
    print("Gerando o .exe de entrega (1 a 2 minutos)...")
    r = subprocess.run([sys.executable, "gerar_exe.py", "--entrega"], capture_output=True, text=True)
    saida = (r.stdout + r.stderr).strip().splitlines()
    print(r.stdout.strip().splitlines()[-1] if r.returncode == 0 else chr(10).join(saida[-30:]))